# PIV-Q27-ACT-v1.8 — Qwen3.8 Reasoning-to-Action Ablation
## ARC-AGI-2 | 4× NVIDIA L4 | Offline Kaggle | Qwen3.8-27B-FP8 TP=4

### Why this experiment exists

The frozen v1.7 XHIGH PIV SMOKE validated the harness but failed to activate the PIV mechanism:

- 8/8 tasks hit 12,288 generated tokens on the first assistant turn;
- 0/8 emitted `</think>`;
- 0/8 executed Python;
- 0/8 produced a final program;
- infrastructure and projected wall-clock were healthy.

Therefore ACT-v1.8 changes **only the reasoning-to-action control**. It does not change the model checkpoint,
PIV prompt, persistent Python workspace, verifier, frozen SMOKE8 tasks, base sampling family, tool-call budget,
or total model-generated token budget.

### Three independent arms, one model load

1. **LOW** — Qwen-native `reasoning_effort="low"`.
2. **MEDIUM** — Qwen-native `reasoning_effort="medium"`.
3. **BUDGET3072** — `reasoning_effort="xhigh"` plus Qwen's documented two-stage thinking-budget technique:
   generate up to 3,072 thinking tokens; if thinking has not closed, append Qwen's documented early-stopping
   text and `</think>`; then continue generation from the exact token prefix.

The completed v1.7 XHIGH SMOKE is a frozen fourth reference and is **not rerun**.

### Pre-registered measurement rule

Every arm reports separately:

- **Thinking behavior:** natural/forced close, thinking tokens, injected-stop tokens, post-close tokens.
- **Activation:** tool-call emission, actual Python execution, tokens/turn to first tool.
- **Tool usefulness:** success/error/timeout/guard rejection, duplicate calls, state reuse.
- **Productivity:** final-program rate, code length/hash diversity.
- **Quality:** all-train-exact, test execution, valid grids, exact scores.
- **Compute:** total tokens, budget death, physical vLLM waves, sec/token, projected Proxy-48 time.
- **Side effects:** forced-close outcome type, retries, parser/interface failures, worker restarts.
- **EDA slices:** tier, dynamic, multi-query, ratio and easy/static segments.
- **Provenance:** arm, effort, think policy, prompt/config/model/runtime fingerprints.

Interpretation is always:

**activation → artifact productivity → executable quality → compute/stability → segment effects**

Tool engagement alone is never treated as solver success.

### Evidence boundary

`PREFLIGHT` proves harness semantics only. It cannot prove how Qwen will behave.
`ACT_SMOKE_GPU` is the real-model measurement.

Do not run Proxy-48 from this notebook. A winning activation policy must first pass the pre-registered SMOKE
gates, then be promoted into a separate frozen PIV Proxy-48 notebook.


In [ ]:
# =====================================================================================
# CELL 1 — PIV-Q27-ACT-v1.8 CONFIG + OFFLINE ENVIRONMENT + FROZEN SOURCE CONTRACT
# Must run before importing torch/vLLM.
# =====================================================================================
import os
import sys
import json
import math
import time
import random
import hashlib
import zipfile
import shutil
import traceback
import copy
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
from collections import deque

SEED = 42
random.seed(SEED)
EXPERIMENT_ID = "PIV-Q27-ACT-v1.8"
GENERATION_CONTRACT_VERSION = "piv-q27-act-agentic-v1.8"
PARSER_CONTRACT_VERSION = "piv-q27-toolxml-finalcode-v1.8"
PROMPT_CONTRACT_VERSION = "piv-q27-source-agentic-prompt-v1.8"

# -------------------------------------------------------------------------------------
# FROZEN SOURCE AUDIT
# -------------------------------------------------------------------------------------
SOURCE_FILE_SHA256 = {
    "qwen_exp1_base": "07d7c7a7ca4567a6118ae8cef00b10277aa37d5ada8698afb66afeb43c88aeb1",
    "qwen_runtime_benchmark": "7d91965196ec3309bf4e3f16f90efcaf2067ee8575d245c8eab9eb100f204b78",
    "arc2_advanced_eda": "1a9384d4b2e01fc15cddad95dee1d6eb5a633b6ae2d041d3d9ae97ffd1879f64",
    "nvarc_31_experiment_diagnostics": "69a64354b6b1c696af1f1c7a68984275d4db9b7eef5f2a382f879dc862d7c978",
    "portability_notes_v10": "c2dd4b45e3697d298d8f814b4087cee742ef4f8632f0eac318ba16cb8f8abda8",
    "experiment_master_plan": "8ca1c34c2c3d864c4b8e04fadf64465cbf468a09b3647c690f7306a07daf0c42"
}

PROXY48_TASK_IDS = [
    "0934a4d8",
    "20a9e565",
    "4c7dc4dd",
    "89565ca0",
    "136b0064",
    "291dc1e1",
    "16b78196",
    "271d71e2",
    "28a6681f",
    "2b83f449",
    "2c181942",
    "31f7f899",
    "35ab12c3",
    "3dc255db",
    "409aa875",
    "4c416de3",
    "64efde09",
    "6e453dd6",
    "7491f3cf",
    "7b0280bc",
    "7b80bb43",
    "7c66cb00",
    "1818057f",
    "332f06d7",
    "53fb4810",
    "7666fa5d",
    "da515329",
    "8f3a5a89",
    "db695cfb",
    "97d7923e",
    "de809cff",
    "c7f57c3e",
    "cb2d8a2c",
    "58490d8a",
    "dbff022c",
    "5961cc34",
    "80a900e0",
    "269e22fb",
    "8b9c3697",
    "e376de54",
    "b99e7126",
    "c4d067a0",
    "135a2760",
    "4a21e3da",
    "71e489b6",
    "800d221b",
    "dd6b8c4b",
    "36a08778"
]
PROXY48_ORDERED_SHA256 = hashlib.sha256("\n".join(PROXY48_TASK_IDS).encode("utf-8")).hexdigest()
PROXY48_EXPECTED_SHA256 = "8f432c60a35091dcda2b890574b1f4d3c03aaf48a5b3b7f96b4e88c35d359134"
PROXY48_SOURCE_COUNTS = {
    "total": 48,
    "tier1": 32,
    "tier2": 12,
    "tier3": 4,
    "easy_static": 16,
    "rare": 30,
    "multi_query": 8,
    "dynamic": 8,
    "hard_tier1": 2,
    "high_ratio": 5,
    "extreme_ratio": 4,
}

if len(PROXY48_TASK_IDS) != 48 or len(set(PROXY48_TASK_IDS)) != 48:
    raise RuntimeError("Frozen Proxy-48 must contain exactly 48 unique task IDs.")
if PROXY48_ORDERED_SHA256 != PROXY48_EXPECTED_SHA256:
    raise RuntimeError("Frozen Proxy-48 fingerprint mismatch. Refusing to run a changed suite.")

# NVARC notebook diagnostic anchors. These are SOURCE diagnostics only, never Q1 results.
NVARC_DIAGNOSTIC_ANCHORS = {
    "outputs_analyzed": 172,
    "gt_present_in_pool": 55,
    "gt_pool_coverage": 55/172,
    "gt_selected_by_current_dual": 45,
    "current_dual_score_reported": 33.50,
    "oracle_score_reported": 41.17,
    "avg_raw_duplicate_ratio": 0.378,
    "scorer_rank_correlation_on_gt_present": 0.961,
    "purpose": "Motivate complementary candidate coverage and provenance-aware dedup; not used to tune Q1 predictions.",
}


# =====================================================================================
# ONE-LINE RUN SWITCH
# =====================================================================================
RUN_PROFILE = "PREFLIGHT"   # "PREFLIGHT" | "ACT_SMOKE_GPU"
RESET_PROXY48_OUTPUTS = False

PIV_SOURCE_COMMIT = "b3d4494892c6d85fa2a35d548271fb02ea3e191b"
PIV_SOURCE_REPO = "https://github.com/gutfeeling/arc-agi-2-submission"
PIV_EXTERNAL_SOURCES = {
    "technical_article": "https://pivotools.github.io/pivotools-quarto-blog/posts/agentic_coding_arc_agi/",
    "audited_repo_snapshot": f"https://github.com/gutfeeling/arc-agi-2-submission/tree/{PIV_SOURCE_COMMIT}",
    "public_experiment_traces": "https://huggingface.co/datasets/arcagi2/arcagi2-agentic-coding-publication",
    "qwen_model_card": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8",
    "qwen_chat_template": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8/blob/main/chat_template.jinja",
    "qwen_thinking_budget_docs": "https://qwen.readthedocs.io/en/latest/getting_started/quickstart.html#thinking-budget",
    "vllm_token_prompt_docs": "https://docs.vllm.ai/en/stable/api/vllm/inputs/",
}
PIV_SOURCE_FILES = {
    "clients.py": {"sha": "34a85ed34001923a5efae715f509e5aa7e8e235f", "role": "persistent model/tool loop + reasoning/tool history"},
    "solver.py": {"sha": "cf2ef554cc389322a5689988b300dc2bc05bf36a", "role": "one code turn -> fresh all-train verification -> test execution"},
    "turn.py": {"sha": "fe4dfc69c5963a56dfba84d0bd49821ae0d2e0bc", "role": "last code block + required solution() artifact"},
    "tools/repl_tool.py": {"sha": "691839335b1e65ba0bd2e4b8b84b70c50d559683", "role": "python tool, errors/timeouts as feedback, puzzle protection"},
    "utils/puzzle_utils.py": {"sha": "7e6ec3034e0dc2f94fba3aa9f1d1d4a0896120b4", "role": "all-train/all-test-input serialization; test outputs hidden"},
    "utils/solving_utils.py": {"sha": "197be842b3d6fa08e4e98066e801dc8cf2bc8327", "role": "fresh sandbox train check and test execution"},
    "utils/code_templates/check_solution_on_training_examples.py": {"sha": "c13ea5b89360ea997a30a1eabe2ccb4a67489b44", "role": "deep-copy input + exact train equality"},
    "solver/prompts/agentic_coding_solver.txt": {"sha": "dab87e08bbe3f071920f13eb98f4a4d6f03ada10", "role": "ARC priors, preloaded puzzle, self-contained solution()"},
    "solver/prompts/system_prompt.txt": {"sha": "265131a50fb42deebe0a8da353f466b11d6d7d6b", "role": "stateful Python tool contract"},
    "solver/config/gpt_oss_120b_high.py": {"sha": "8fd563ddcb10962c7564fbfa8cbf3424a7a437d7", "role": "source agentic config example"},
    "sandbox/ipybox_sandbox.py": {"sha": "195e0e66ad7328a4f823974f04778b639a16412a", "role": "source Docker/IPyBox persistent REPL implementation; semantics only, not ported literally"},
    "sandbox/base.py": {"sha": "517f5794ff0cac64024e47c184d116b1f36577ea", "role": "source stateful REPL/Sandbox interface and fresh run_cells semantics"},
    "evaluation/evaluate.py": {"sha": "8b48fb4482dc7d9772ac010ba6dd81f6cc15e0a4", "role": "source multi-sample evaluation / majority aggregation; not copied into P0"},
    "utils/code_templates/produce_output_grid.py": {"sha": "b6ab435ca4f7d77414764e3a4400bb8c6dbead5e", "role": "execute verified solution on all test inputs"},
}

PROFILE_PROFILE_TABLE = {
    "PREFLIGHT": {
        "use_model": False, "task_set": "SMOKE8", "active_tasks": 8,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_ACT_V18_PREFLIGHT_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": False,
    },
    "ACT_SMOKE_GPU": {
        "use_model": True, "task_set": "SMOKE8", "active_tasks": 8,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_ACT_V18_SMOKE_GPU_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": False,
    },
}
RUN_PROFILE = str(RUN_PROFILE).upper()
if RUN_PROFILE not in PROFILE_TABLE:
    raise ValueError(f"RUN_PROFILE={RUN_PROFILE!r} invalid; choose {list(PROFILE_TABLE)}")
PROFILE = PROFILE_TABLE[RUN_PROFILE]
USE_MODEL = bool(PROFILE["use_model"])
TASK_SET = str(PROFILE["task_set"])
ACTIVE_TASK_LIMIT = int(PROFILE["active_tasks"])
AGENT_BATCH_SIZE = int(PROFILE["agent_batch_size"])
MAX_TOOL_CALLS = int(PROFILE["max_tool_calls"])
MAX_MODEL_TURNS = int(PROFILE["max_model_turns"])
MAX_RETRIES = int(PROFILE.get("max_retries", 2))
PER_TURN_MAX_TOKENS = int(PROFILE["per_turn_max_tokens"])
TOTAL_GENERATED_TOKEN_CAP = int(PROFILE["total_generated_token_cap"])
TOOL_TIMEOUT_SEC = float(PROFILE["tool_timeout_sec"])
VERIFIER_TIMEOUT_SEC = float(PROFILE["verifier_timeout_sec"])
BUNDLE_NAME = str(PROFILE["bundle_name"])
ACCURACY_IS_REAL = bool(PROFILE["accuracy_is_real"])

# =====================================================================================
# ACT-v1.8 PRE-REGISTERED REASONING-TO-ACTION ARMS
# =====================================================================================
ACT_ARM_ORDER = ["LOW", "MEDIUM", "BUDGET3072"]
ACT_ARMS = {
    "LOW": {
        "reasoning_effort": "low",
        "think_policy": "native_effort",
        "thinking_budget_tokens": None,
        "seed_offset": 0,
    },
    "MEDIUM": {
        "reasoning_effort": "medium",
        "think_policy": "native_effort",
        "thinking_budget_tokens": None,
        "seed_offset": 0,
    },
    "BUDGET3072": {
        "reasoning_effort": "xhigh",
        "think_policy": "qwen_two_stage_budget",
        "thinking_budget_tokens": 3072,
        "seed_offset": 0,
    },
}
# Same stochastic seed across arms by design: the causal difference is the reasoning-to-action policy.
ACT_BASE_SEED = SEED + 3000

# Exact Qwen-documented early-stopping text for open-source two-stage thinking-budget control.
# The injected tokens are NOT model-generated tokens. They are logged separately as context/prefill cost.
QWEN_EARLY_STOPPING_TEXT = (
    "\n\n Considering the limited time by the user, I have to give the solution based on the thinking directly now."
    "\n</think>\n\n"
)

# Frozen external reference: the already-completed v1.7 XHIGH SMOKE, not rerun in this notebook.
FROZEN_XHIGH_SMOKE_REFERENCE = {
    "bundle_sha256": "7435b795691d1542100f7f5cd0764ebcdacc61be6128f236c845f10eb22b143f",
    "tasks": 8,
    "tool_engagement_rate": 0.0,
    "final_program_rate": 0.0,
    "all_train_exact_rate": 0.0,
    "budget_death_rate": 1.0,
    "first_turn_length_rate": 1.0,
    "generated_tokens_per_task": 12288,
    "wave_sec": 1967.8,
    "projected_proxy48_total_hours": 3.40,
}

MEASUREMENT_CONTRACT = {
    "version": "piv-q27-act-measurement-v1",
    "causal_variable": "reasoning_to_action_control",
    "frozen_across_arms": [
        "model_checkpoint","TP4_engine","SMOKE8_task_ids_and_order","agentic_prompt","python_tool_schema",
        "persistent_REPL","fresh_exact_train_verifier","sampling_temperature_top_p_top_k_min_p_penalties",
        "max_6_executed_tool_calls","max_12_logical_model_turns","12288_model_generated_tokens_per_task",
        "tool_and_verifier_timeouts","grid_validation","seed_base"
    ],
    "arm_specific": {
        "LOW": "reasoning_effort=low",
        "MEDIUM": "reasoning_effort=medium",
        "BUDGET3072": "reasoning_effort=xhigh + documented two-stage forced close at 3072 thinking tokens"
    },
    "metric_families": {
        "thinking_behavior": [
            "natural_think_close_count","forced_think_close_count","think_generated_tokens",
            "injected_stop_tokens","post_close_generated_tokens","tokens_to_first_tool"
        ],
        "activation": [
            "tool_call_emitted_rate","tool_engagement_rate","first_tool_turn","final_program_rate"
        ],
        "tool_usefulness": [
            "tool_successes","tool_errors","tool_timeouts","guard_rejections","repeated_tool_calls",
            "unique_tool_code_count","stateful_reuse_events","stateful_reused_symbols"
        ],
        "program_productivity": [
            "final_program_rate","syntax_valid_final_rate","final_code_length","final_code_hash_diversity"
        ],
        "program_quality": [
            "all_train_exact_rate","test_execution_ok_rate","valid_grid_rate","output_weighted_exact",
            "task_normalized_exact"
        ],
        "budget_compute": [
            "generated_tokens","injected_stop_tokens","budget_death_rate","tokens_at_death","wave_sec",
            "sec_per_output_token","projected_proxy48_total_hours","projected_proxy48_total_hours_20pct_margin"
        ],
        "interface_failures": [
            "malformed_tool_xml","invalid_finish_reason","no_tool_or_final","worker_restarts",
            "source_retries","step_retries","budget_denials"
        ],
        "diversity": [
            "final_code_hash_diversity","candidate_grid_hash_diversity","duplicate_candidate_ratio"
        ],
        "segments": [
            "tier1","tier2","tier3","easy_static","rare","multi_query","dynamic","high_ratio","extreme_ratio"
        ],
    },
    "activation_promotion_gate": {
        "terminal_coverage_complete": True,
        "infrastructure_failure_rate": 0.0,
        "budget_death_rate_max": 0.50,
        "tool_engagement_rate_min": 0.50,
        "final_program_rate_min": 0.25,
        "projected_proxy48_total_hours_max": 10.0,
        "all_train_exact_rate": "diagnostic strong-green, not mandatory on SMOKE8"
    },
    "interpretation_rule": "Report activation -> productivity -> quality -> compute separately; never equate tool engagement with solver quality.",
}

MODEL_PATH = Path("/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1")
WHEELHOUSE = Path("/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3")
ARC_ROOT = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
TRAIN_CHALLENGES_PATH = ARC_ROOT / "arc-agi_training_challenges.json"
TRAIN_SOLUTIONS_PATH  = ARC_ROOT / "arc-agi_training_solutions.json"
EVAL_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS_PATH   = ARC_ROOT / "arc-agi_evaluation_solutions.json"
TEST_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_test_challenges.json"
SAMPLE_SUBMISSION_PATH = ARC_ROOT / "sample_submission.json"

BASE_WORK_ROOT = Path("/kaggle/working/piv_q27_act_v18")
WORK_ROOT = BASE_WORK_ROOT / RUN_PROFILE.lower()
if PROFILE["reset_outputs"] and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
with open(WORK_ROOT/"MEASUREMENT_CONTRACT.json","w",encoding="utf-8") as _f:
    json.dump(MEASUREMENT_CONTRACT,_f,indent=2)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NOTEBOOK_WALL_START = time.perf_counter()
TOTAL_RUNTIME_SEC = 12 * 60 * 60
# v1.6: one wave may run for the remaining P0 budget (up to 12,288 tokens ~= 33 min at the Q1-measured
# ~0.16 s/step). The reserve must exceed the longest possible wave plus bundle time, and each wave is
# additionally pre-checked against a projection built from measured wave timings.
SAFETY_RESERVE_SEC = 60 * 60
DEFAULT_SEC_PER_OUTPUT_TOKEN = 0.20   # projection prior before the first measured wave
WAVE_TIME_SAFETY_FACTOR = 1.25
WAVE_OVERHEAD_SEC = 90.0              # prefill / scheduling / tool + verifier allowance per wave

# Frozen from the successful Q1 4×L4 runtime.
MAX_MODEL_LEN = 29696
MAX_NUM_BATCHED_TOKENS = 32768
MAX_NUM_SEQS = 16
GPU_MEMORY_UTILIZATION = 0.90
# v1.6 LOCAL throughput option (algorithm-neutral): prefix caching lets every multi-turn prompt reuse the
# already-computed conversation prefix instead of re-prefilling it each turn. Default False keeps the
# Q1-frozen engine fingerprint. Set True after SMOKE if the projected wall-clock is tight; if vLLM refuses
# it for this hybrid-attention checkpoint, revert to False.
ENABLE_PREFIX_CACHING = False

# EXP-0 diagnostics preserved unchanged.
WEIGHT_CLIP_LOW = 0.20
WEIGHT_CLIP_HIGH = 5.00
N_PROPENSITY_FOLDS = 5
HIGH_RATIO_THRESHOLD = 10.0
LARGE_TEST_INPUT_CELLS = 300
MANY_COLORS_THRESHOLD = 7
LOW_BG_THRESHOLD = 0.30

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("=" * 116)
print(EXPERIMENT_ID, "| Stateful Python REPL + Fresh Exact Train Verifier")
print("=" * 116)
print("RUN_ID:", RUN_ID, "| RUN_PROFILE:", RUN_PROFILE, "| TASK_SET:", TASK_SET)
print("PIV source commit:", PIV_SOURCE_COMMIT)
print("Agent budget: calls/turns/per-turn/total =", MAX_TOOL_CALLS, MAX_MODEL_TURNS, PER_TURN_MAX_TOKENS, TOTAL_GENERATED_TOKEN_CAP)
print("Proxy-48 SHA256:", PROXY48_ORDERED_SHA256)
print("WORK_ROOT:", WORK_ROOT)

required = [MODEL_PATH, WHEELHOUSE, ARC_ROOT, TRAIN_CHALLENGES_PATH, TRAIN_SOLUTIONS_PATH,
            EVAL_CHALLENGES_PATH, EVAL_SOLUTIONS_PATH, TEST_CHALLENGES_PATH, SAMPLE_SUBMISSION_PATH]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Required Kaggle input is missing: {p}")
config_path = MODEL_PATH / "config.json"
index_path = MODEL_PATH / "model.safetensors.index.json"
with open(config_path, "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
qcfg = model_cfg.get("quantization_config", {})
if model_cfg.get("model_type") != "qwen3_5":
    raise RuntimeError(f"Unexpected model_type={model_cfg.get('model_type')}")
if qcfg.get("quant_method") != "fp8" or qcfg.get("fmt") != "e4m3":
    raise RuntimeError(f"Unexpected quantization_config={qcfg}")

def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

def stable_json_sha256(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def small_file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_RUNTIME_PACKAGES = {"vllm":"0.19.0", "transformers":"4.57.6", "tokenizers":"0.22.2", "huggingface-hub":"0.36.2"}
_loaded_module_names = {"vllm":"vllm", "transformers":"transformers", "tokenizers":"tokenizers", "huggingface-hub":"huggingface_hub"}
before_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
for _pkg, _module in _loaded_module_names.items():
    if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _module in sys.modules:
        raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart Kaggle session before Run All.")

def offline_pip_install(requirements):
    if not requirements: return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])
if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
    offline_pip_install(["vllm==0.19.0"])
else:
    repair=[]
    for package in ["transformers","tokenizers","huggingface-hub"]:
        if before_stack[package] != EXPECTED_RUNTIME_PACKAGES[package]:
            module_name = package.replace("-", "_")
            if module_name in sys.modules or (package=="huggingface-hub" and "huggingface_hub" in sys.modules):
                raise RuntimeError(f"{package} already imported with wrong version; restart session.")
            repair.append(f"{package}=={EXPECTED_RUNTIME_PACKAGES[package]}")
    offline_pip_install(repair)
after_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
for package, wanted in EXPECTED_RUNTIME_PACKAGES.items():
    if after_stack.get(package) != wanted:
        raise RuntimeError(f"Runtime parity failure: {package}={after_stack.get(package)!r}, expected {wanted!r}")
print("Runtime dependency parity PASS:", after_stack)


In [ ]:

# =====================================================================================
# CELL 2 — LOAD ARC DATA
# =====================================================================================
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(SEED)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_challenges = load_json(TRAIN_CHALLENGES_PATH)
train_solutions  = load_json(TRAIN_SOLUTIONS_PATH)
eval_challenges  = load_json(EVAL_CHALLENGES_PATH)
eval_solutions   = load_json(EVAL_SOLUTIONS_PATH)
test_challenges  = load_json(TEST_CHALLENGES_PATH)
sample_submission = load_json(SAMPLE_SUBMISSION_PATH)

assert set(eval_challenges) == set(eval_solutions)
assert set(test_challenges) == set(sample_submission)
missing_proxy_ids = [k for k in PROXY48_TASK_IDS if k not in eval_challenges or k not in eval_solutions]
if missing_proxy_ids:
    raise RuntimeError(f"Frozen Proxy-48 IDs missing from public Evaluation data: {missing_proxy_ids}")

print("Training tasks       :", len(train_challenges))
print("Evaluation tasks     :", len(eval_challenges))
print("Evaluation outputs   :", sum(len(v) for v in eval_solutions.values()))
print("Test tasks           :", len(test_challenges))
print("Visible test queries :", sum(len(v["test"]) for v in test_challenges.values()))


In [ ]:

# =====================================================================================
# CELL 3 — EXP-0 STRUCTURAL FEATURES + TEST-RELEVANCE WEIGHTS
# Recomputed here so EXP-1 is self-contained.
# =====================================================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, log_loss

def arr(grid):
    return np.asarray(grid, dtype=np.int16)

def is_rectangular_grid(grid, max_hw=30):
    if not isinstance(grid, list) or len(grid) < 1 or len(grid) > max_hw:
        return False
    if not all(isinstance(row, list) for row in grid):
        return False
    widths = [len(row) for row in grid]
    if min(widths, default=0) < 1 or max(widths, default=0) > max_hw or len(set(widths)) != 1:
        return False
    for row in grid:
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, np.integer)):
                return False
            if int(x) < 0 or int(x) > 9:
                return False
    return True

def grid_basic_stats(grid):
    a = arr(grid)
    vals, counts = np.unique(a, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-(probs * np.log(probs + 1e-12)).sum())
    return {
        "h": int(a.shape[0]),
        "w": int(a.shape[1]),
        "cells": int(a.size),
        "n_colors": int(len(vals)),
        "bg0_density": float(np.mean(a == 0)),
        "color_entropy": entropy,
    }

def colored_components_count(grid):
    # Diagnostic only: 4-connected same-color non-zero components.
    a = arr(grid)
    h, w = a.shape
    seen = np.zeros((h, w), dtype=bool)
    n = 0
    sizes = []
    for r in range(h):
        for c in range(w):
            if seen[r, c] or a[r, c] == 0:
                continue
            color = a[r, c]
            q = deque([(r, c)])
            seen[r, c] = True
            size = 0
            while q:
                rr, cc = q.popleft()
                size += 1
                for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
                    nr, nc = rr+dr, cc+dc
                    if 0 <= nr < h and 0 <= nc < w and not seen[nr, nc] and a[nr, nc] == color:
                        seen[nr, nc] = True
                        q.append((nr, nc))
            n += 1
            sizes.append(size)
    return n, (float(np.mean(sizes)) if sizes else 0.0)

def summarize(values, prefix):
    x = np.asarray(values, dtype=float)
    if len(x) == 0:
        return {
            f"{prefix}_mean": 0.0, f"{prefix}_std": 0.0,
            f"{prefix}_min": 0.0, f"{prefix}_max": 0.0,
        }
    return {
        f"{prefix}_mean": float(x.mean()),
        f"{prefix}_std": float(x.std()),
        f"{prefix}_min": float(x.min()),
        f"{prefix}_max": float(x.max()),
    }

def task_features(task_id, task, split):
    train = task["train"]
    tests = task["test"]

    tr_in_stats, tr_out_stats, te_in_stats = [], [], []
    tr_in_components, tr_out_components, te_components = [], [], []
    tr_in_component_sizes, tr_out_component_sizes, te_component_sizes = [], [], []
    ratios, h_ratios, w_ratios, same_shape = [], [], [], []
    all_visible_colors = set()

    for ex in train:
        si = grid_basic_stats(ex["input"])
        so = grid_basic_stats(ex["output"])
        tr_in_stats.append(si); tr_out_stats.append(so)
        ci, csi = colored_components_count(ex["input"])
        co, cso = colored_components_count(ex["output"])
        tr_in_components.append(ci); tr_out_components.append(co)
        tr_in_component_sizes.append(csi); tr_out_component_sizes.append(cso)
        ratios.append(so["cells"] / max(si["cells"], 1))
        h_ratios.append(so["h"] / max(si["h"], 1))
        w_ratios.append(so["w"] / max(si["w"], 1))
        same_shape.append(int((si["h"], si["w"]) == (so["h"], so["w"])))
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())
        all_visible_colors.update(np.unique(arr(ex["output"])).tolist())

    for ex in tests:
        st = grid_basic_stats(ex["input"])
        te_in_stats.append(st)
        ct, cst = colored_components_count(ex["input"])
        te_components.append(ct); te_component_sizes.append(cst)
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())

    feat = {
        "task_id": task_id,
        "split": split,
        "n_train": len(train),
        "n_test_queries": len(tests),
        "visible_n_colors": len(all_visible_colors),
        "same_shape_frac": float(np.mean(same_shape)),
        "dynamic_any": int(any(x == 0 for x in same_shape)),
        "multi_query": int(len(tests) > 1),
    }

    for key in ["h", "w", "cells", "n_colors", "bg0_density", "color_entropy"]:
        feat.update(summarize([x[key] for x in tr_in_stats], f"tr_in_{key}"))
        feat.update(summarize([x[key] for x in tr_out_stats], f"tr_out_{key}"))
        feat.update(summarize([x[key] for x in te_in_stats], f"te_in_{key}"))

    feat.update(summarize(ratios, "area_ratio"))
    feat.update(summarize(h_ratios, "h_ratio"))
    feat.update(summarize(w_ratios, "w_ratio"))
    feat.update(summarize(tr_in_components, "tr_in_components"))
    feat.update(summarize(tr_out_components, "tr_out_components"))
    feat.update(summarize(te_components, "te_in_components"))
    feat.update(summarize(tr_in_component_sizes, "tr_in_component_size"))
    feat.update(summarize(tr_out_component_sizes, "tr_out_component_size"))
    feat.update(summarize(te_component_sizes, "te_in_component_size"))

    feat["n_train_eq_2"] = int(len(train) == 2)
    feat["n_train_ge_3"] = int(len(train) >= 3)
    feat["high_ratio"] = int(max(ratios) >= HIGH_RATIO_THRESHOLD)
    feat["large_test_input"] = int(max(x["cells"] for x in te_in_stats) >= LARGE_TEST_INPUT_CELLS)
    feat["many_colors"] = int(len(all_visible_colors) >= MANY_COLORS_THRESHOLD)
    feat["low_bg_train_input"] = int(
        np.mean([x["bg0_density"] for x in tr_in_stats]) < LOW_BG_THRESHOLD
    )
    return feat

def build_feature_df(data, split):
    return pd.DataFrame(
        [task_features(k, v, split) for k, v in data.items()]
    ).sort_values("task_id").reset_index(drop=True)

train_feat = build_feature_df(train_challenges, "train")
eval_feat  = build_feature_df(eval_challenges, "eval")
test_feat  = build_feature_df(test_challenges, "test")

for name, df in [("train", train_feat), ("eval", eval_feat), ("test", test_feat)]:
    df.to_csv(WORK_ROOT / f"task_features_{name}.csv", index=False)

numeric_cols = [
    c for c in eval_feat.columns
    if c not in {"task_id", "split"} and pd.api.types.is_numeric_dtype(eval_feat[c])
]
EXCLUDE_PROPENSITY = {
    "task_id", "split",
    "n_train_eq_2", "n_train_ge_3", "high_ratio", "large_test_input",
    "many_colors", "low_bg_train_input", "multi_query",
}
prop_cols = [c for c in numeric_cols if c not in EXCLUDE_PROPENSITY]

combined = pd.concat(
    [eval_feat.assign(_is_test=0), test_feat.assign(_is_test=1)],
    ignore_index=True
)
X = combined[prop_cols].astype(float)
y = combined["_is_test"].astype(int).to_numpy()

prop_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        C=0.5, penalty="l2", solver="liblinear",
        max_iter=5000, random_state=SEED
    )),
])
cv = StratifiedKFold(n_splits=N_PROPENSITY_FOLDS, shuffle=True, random_state=SEED)
p_oof = cross_val_predict(prop_model, X, y, cv=cv, method="predict_proba")[:, 1]

prop_auc = roc_auc_score(y, p_oof)
prop_logloss = log_loss(y, p_oof)

n_eval = len(eval_feat)
n_test = len(test_feat)
prior_eval = n_eval / (n_eval + n_test)
prior_test = n_test / (n_eval + n_test)

p_eval_as_test = np.clip(p_oof[:n_eval], 1e-4, 1 - 1e-4)
density_raw = (p_eval_as_test / (1 - p_eval_as_test)) * (prior_eval / prior_test)
density_clip = np.clip(density_raw, WEIGHT_CLIP_LOW, WEIGHT_CLIP_HIGH)
density_norm = density_clip / np.mean(density_clip)

eval_weights = eval_feat[["task_id"]].copy()
eval_weights["p_test_oof"] = p_eval_as_test
eval_weights["density_ratio_raw"] = density_raw
eval_weights["weight"] = density_norm
eval_weights["test_relevance_rank_pct"] = pd.Series(density_norm).rank(pct=True).to_numpy()
eval_weights["test_relevance_quartile"] = pd.qcut(
    pd.Series(density_norm).rank(method="first"),
    q=4,
    labels=["Q1_low", "Q2", "Q3", "Q4_high"]
).astype(str).to_numpy()

weight_ess = float((density_norm.sum() ** 2) / np.sum(density_norm ** 2))
eval_weights.to_csv(WORK_ROOT / "eval_test_propensity_weights.csv", index=False)

print("EXP-0 reproduction:")
print("  Eval-vs-Test CV AUC :", round(prop_auc, 6))
print("  Log loss            :", round(prop_logloss, 6))
print("  Weight ESS          :", round(weight_ess, 3), "/", len(eval_weights))
print("  Weight min/median/max:",
      round(float(density_norm.min()), 4),
      round(float(np.median(density_norm)), 4),
      round(float(density_norm.max()), 4))


# =====================================================================================
# SOURCE-ALIGNED PROXY-48 FEATURES
# Exact feature/tag rules copied from the source EDA notebook for audit comparability.
# These DO NOT replace EXP-0 features or official scoring.
# =====================================================================================
def source_eda_n_cells(grid):
    return int(np.prod(np.shape(grid)))

def source_eda_proxy_features(task_id, data):
    puzzle = data[task_id]
    train = puzzle["train"]
    input_sizes = [source_eda_n_cells(ex["input"]) for ex in train]
    output_sizes = [source_eda_n_cells(ex["output"]) for ex in train]
    ratios = [max(i, o) / max(min(i, o), 1) for i, o in zip(input_sizes, output_sizes)]
    colors = set()
    pixels = []
    for ex in train:
        for field in ["input", "output"]:
            a = np.asarray(ex[field])
            colors.update(np.unique(a).tolist())
            pixels.extend(a.reshape(-1).tolist())

    avg_input_size = float(np.mean(input_sizes))
    avg_ratio = float(np.mean(ratios))
    max_ratio = float(max(ratios))
    n_colors = len(colors)
    n_train = len(train)
    n_queries = len(puzzle["test"])
    is_dynamic = any(np.shape(ex["input"]) != np.shape(ex["output"]) for ex in train)
    is_rare = bool(6 in colors or 9 in colors)
    bg_density = pixels.count(0) / max(len(pixels), 1)

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900 * 0.30 +
        (avg_ratio - 1) / 10 * 0.25 +
        n_colors / 10 * 0.20 +
        (0.25 if is_dynamic else 0.0)
    )
    if n_train >= 6 and tier < 3:
        fine *= 0.7

    tags = [f"tier{tier}"]
    if tier == 1 and (not is_dynamic) and max_ratio == 1 and n_colors <= 5:
        tags.append("easy_static")
    if tier == 1 and fine <= 0.22:
        tags.append("easy")
    if tier == 1 and fine > 0.45:
        tags.append("hard_tier1")
    if is_dynamic:
        tags.append("dynamic")
    if is_rare:
        tags.append("rare")
    if n_queries > 1:
        tags.append("multi_query")
    if max_ratio > 5:
        tags.append("high_ratio")
    if max_ratio > 30:
        tags.append("extreme_ratio")
    if n_colors >= 8:
        tags.append("many_colors")
    if avg_input_size >= 500:
        tags.append("large")
    if bg_density >= 0.70:
        tags.append("sparse_bg")
    if bg_density <= 0.30:
        tags.append("dense_non_bg")
    if n_train <= 2:
        tags.append("low_train")

    return {
        "task_id": task_id,
        "proxy_tier": int(tier),
        "proxy_fine": float(fine),
        "proxy_avg_input_size": avg_input_size,
        "proxy_max_ratio": max_ratio,
        "proxy_avg_ratio": avg_ratio,
        "proxy_n_colors": int(n_colors),
        "proxy_dynamic": int(is_dynamic),
        "proxy_rare": int(is_rare),
        "proxy_n_train": int(n_train),
        "proxy_n_queries": int(n_queries),
        "proxy_bg_density": float(bg_density),
        "proxy_easy_static": int("easy_static" in tags),
        "proxy_hard_tier1": int("hard_tier1" in tags),
        "proxy_high_ratio": int("high_ratio" in tags),
        "proxy_extreme_ratio": int("extreme_ratio" in tags),
        "proxy_tags": "|".join(tags),
    }

proxy_source_feat = pd.DataFrame([
    source_eda_proxy_features(k, eval_challenges) for k in eval_challenges
]).sort_values("task_id").reset_index(drop=True)
proxy_source_feat.to_csv(WORK_ROOT / "source_eda_features_eval.csv", index=False)

proxy48_source = proxy_source_feat[proxy_source_feat["task_id"].isin(PROXY48_TASK_IDS)].copy()
proxy48_order = {k: i for i, k in enumerate(PROXY48_TASK_IDS)}
proxy48_source["_order"] = proxy48_source["task_id"].map(proxy48_order)
proxy48_source = proxy48_source.sort_values("_order").drop(columns="_order").reset_index(drop=True)
proxy48_source.to_csv(WORK_ROOT / "proxy48_source_eda.csv", index=False)

proxy_source_counts = {
    "total": int(len(proxy48_source)),
    "tier1": int((proxy48_source["proxy_tier"] == 1).sum()),
    "tier2": int((proxy48_source["proxy_tier"] == 2).sum()),
    "tier3": int((proxy48_source["proxy_tier"] == 3).sum()),
    "easy_static": int(proxy48_source["proxy_easy_static"].sum()),
    "rare": int(proxy48_source["proxy_rare"].sum()),
    "multi_query": int((proxy48_source["proxy_n_queries"] > 1).sum()),
    "dynamic": int(proxy48_source["proxy_dynamic"].sum()),
    "hard_tier1": int(proxy48_source["proxy_hard_tier1"].sum()),
    "high_ratio": int(proxy48_source["proxy_high_ratio"].sum()),
    "extreme_ratio": int(proxy48_source["proxy_extreme_ratio"].sum()),
}
if proxy_source_counts != PROXY48_SOURCE_COUNTS:
    raise RuntimeError(
        "Proxy-48 source-EDA profile drift detected. "
        f"expected={PROXY48_SOURCE_COUNTS}, actual={proxy_source_counts}"
    )
with open(WORK_ROOT / "proxy48_source_distribution.json", "w", encoding="utf-8") as f:
    json.dump(proxy_source_counts, f, indent=2)
print("Proxy-48 source-EDA contract: PASS", proxy_source_counts)


In [ ]:
# =====================================================================================
# CELL 4 — TOKENIZER + SOURCE-ALIGNED AGENTIC PROMPT + QWEN TOOL TEMPLATE + ACTIVE TASKS
# =====================================================================================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True, trust_remote_code=False)
print("Tokenizer:", tokenizer.__class__.__name__, "| vocab:", len(tokenizer))

# OpenAI-style schema; Qwen3.8's official chat template serializes it into its XML tool-call contract.
PYTHON_TOOL = {
    "type": "function",
    "function": {
        "name": "python",
        "description": "Execute Python in a persistent per-puzzle REPL and return stdout or a concise error.",
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string", "description": "Python code to execute"}},
            "required": ["code"],
            "additionalProperties": False,
        },
    },
}
TOOL_SCHEMA_SHA256 = stable_json_sha256(PYTHON_TOOL)

SOURCE_SYSTEM_PROMPT = '# Tools\n\n## python\n\nUse this tool to execute Python code in your chain of thought. \n\nWhen you send a message containing Python code to python, it will be executed in a stateful Jupyter notebook environment. The environment supports common third party libraries like numpy, scipy, shapely, networkx, scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver. python will respond with the output of the execution or time out after 120.0 seconds. \n\n### Usage guideline\n\n1. Use the variables and functions already loaded in the Jupyter notebook instead of defining them yourself. The user message will provide a complete list of such variables and functions.\n2. Prefer to write reusable functions. These will persist in the stateful Jupyter notebook and can be referenced in later code snippets without having to define them again.'
SOURCE_AGENTIC_PROMPT_TEMPLATE = 'You are an expert ARC AGI puzzle solver. Please solve the puzzle given below.\n\n# Background information about ARC AGI puzzles\n\nEach task consists of a small number of demonstration examples (3.3 on average), and a small number of test examples (1 - 3).\n\nEach example consists of an “input grid” and an “output grid”. Each “grid” is a literal grid of colored squares There are 10 unique colors. A grid can be any height or width between 1x1 and 30x30, inclusive.\n\nThe color code is as follows:\n\n0 - black\n1 - navy blue\n2 - red\n3 - green\n4 - yellow\n5 - gray\n6 - pink\n7 - orange\n8 - sky blue\n9 - maroon\n\nWhen solving an evaluation task, a test-taker has access to the training examples for the task (both the input and output grids), as well as the input grid of the test examples for the task. The test-taker must construct on its own the output grid corresponding to the input grid of each test example. “Constructing the output grid” is done entirely from scratch, meaning that the test-taker must decide what the height and width of the output grid should be, what colors they should place on the grid, and where. The task is successfully solved if the test-taker can produce the exact correct answer on all test examples for the task (binary measure of success).\n\nThe mapping from integers (0-9) to color names is arbitrary and should be treated as categorical labels; the transformation rule never relies on the specific integer–color pairing.\n\nARC AGI puzzle solutions feel "natural" to humans. Natural rules tend to build on visual priors and small-number counting. They are visually immediate or prominent for humans. That is why humans from all educational backgrounds are able to solve these puzzles. Rules that depend on non-visual considerations (e.g. mathematics that\'s more advanced than small number counting) or inelegant mechanics (e.g. tie-breakers, complex conditional rules, special exceptions) are considered "unnatural."  \n\nThis doesn\'t mean puzzles have to be simple or easy. Common patterns that puzzles use to increase difficulty are:  \n\n- **Rule composition:** multiple simple, natural rules are applied in sequence, with one transformation setting up the next.\n- **Contextual application:** a simple if/else based on a visually clear feature.\n- **In-grid symbol definition:** certain shapes or patterns act as symbols whose meaning is established using the training examples and which control some aspect of the transformation. The meaning can even vary across training examples.\n- **Iterated steps:** a simple action is applied repeatedly (as in assembling pieces of a jigsaw), where each step constrains the next. Repetition is restricted to a small number.\n- **Approximate features**: Features that are easily detected by humans, but hard to detect using naive algorithms. Examples are slightly broken symmetry, slightly broken continuity, off-center symmetry axes, objects defined via 8-connectivity etc.\n- **Non-locality:** the solution requires applying changes to one region, with the changes depending on features in a different, distant region.\n\n# Puzzle to solve\n\n{{puzzle}}\n\n# Task aids\n\nYou may execute code in the Jupyter notebook environment.\n\nVariables available in the Jupyter notebook:\n\n- `puzzle`: A Python `dict` containing the training and test grids. The key "train" contains the training examples in a list. Similarly, the key "test" contains the test examples as a list. Each training/test example is a dict containing keys "input" and "output". The values are the grids expressed as list of lists of integers (0 - 9). Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.\n\nYour very first code execution should be `puzzle.keys()` to check data access. \n\nDon\'t print the `puzzle` dict, because it prints out the grids that are already present in this user message and consumes valuable token budget.\n\n# Your task\n\nPlease solve the puzzle. Solving the puzzle means constructing a function `solution(grid: list[list[int]])` that implements the transformation from input grid to output grid shown in the training examples. The solution must work on all training examples.\n\n```\nfor idx, example in enumerate(puzzle["train"]):\n    assert solution(example["input"]) == example["output"], f"Train example {idx}: mismatch"\n```\n\nThe solution must generalize to the test input(s). Solutions that cheat by hardcoding the training example outputs are not acceptable.\n\n# Output format (STRICT)\n\nYour response must contain a single Python code block fenced by ```python ... ```. Must include a `solution` function. Must be self-contained i.e. include all necessary imports and helper functions.'

# LOCAL adaptation: source advertises a broad dependency list and 120s IPython timeout.
# The portability audit recommends a smaller P0 stack. Probe the actual current Python
# environment and advertise only approved libraries that really import.
import subprocess as _piv_subprocess
P0_LIBRARY_PROBES = [
    ("numpy","numpy"), ("scipy","scipy"), ("scikit-image","skimage"),
    ("networkx","networkx"), ("Pillow","PIL"),
]
AVAILABLE_P0_LIBRARIES=[]
P0_LIBRARY_PROBE_RESULTS={}
for _label,_module in P0_LIBRARY_PROBES:
    try:
        _cp=_piv_subprocess.run(
            [sys.executable,"-c",f"import {_module}"],
            text=True,capture_output=True,timeout=8,
        )
        _ok=(_cp.returncode==0)
        P0_LIBRARY_PROBE_RESULTS[_label]={"ok":_ok,"stderr":(_cp.stderr or "")[-500:]}
        if _ok: AVAILABLE_P0_LIBRARIES.append(_label)
    except Exception as _e:
        P0_LIBRARY_PROBE_RESULTS[_label]={"ok":False,"stderr":f"{type(_e).__name__}: {_e}"}

_source_lib_sentence=(
    "The environment supports common third party libraries like numpy, scipy, shapely, networkx, "
    "scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver."
)
_local_lib_sentence=(
    "For this LOCAL P0 port, approved third party libraries verified in the current environment are: "
    + (", ".join(AVAILABLE_P0_LIBRARIES) if AVAILABLE_P0_LIBRARIES else "none beyond the Python standard library")
    + ". Do not assume optional source libraries such as OR-Tools, z3-solver, python-constraint, or shapely are available."
)
SYSTEM_PROMPT = SOURCE_SYSTEM_PROMPT.replace(_source_lib_sentence,_local_lib_sentence).replace(
    "time out after 120.0 seconds",
    f"time out after {TOOL_TIMEOUT_SEC:.1f} seconds",
)
SYSTEM_PROMPT += (
    f"\n\n### Local P0 compute budget\n"
    f"You may execute Python at most {MAX_TOOL_CALLS} times in this task. "
    "Use the calls deliberately. When the final Python call has been used, do not request another tool call; "
    "return the self-contained final `solution(grid)` code block on the next assistant turn."
)

def grid_lines(grid):
    return "\n".join(" ".join(str(int(x)) for x in row) for row in grid)

def puzzle_to_source_text(task):
    parts = ["## Training examples"]
    for i, ex in enumerate(task["train"]):
        parts += [f"### Training example {i}", "Input:", grid_lines(ex["input"]), "Output:", grid_lines(ex["output"])]
    parts.append("## Tests" if len(task["test"]) > 1 else "## Test")
    for i, ex in enumerate(task["test"]):
        if len(task["test"]) > 1:
            parts.append(f"### Test {i}")
        parts += ["Input:", grid_lines(ex["input"])]
    return "\n\n".join(parts)

def build_agent_user_prompt(task):
    # Keep the source prompt intact except for two necessary local substitutions:
    # 1) serialize the current puzzle; 2) clarify that test outputs are absent from the preloaded variable.
    prompt = SOURCE_AGENTIC_PROMPT_TEMPLATE.replace("{{puzzle}}", puzzle_to_source_text(task))
    prompt = prompt.replace(
        'Each training/test example is a dict containing keys "input" and "output". '
        'The values are the grids expressed as list of lists of integers (0 - 9). '
        'Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.',
        'Each training example is a dict containing keys "input" and "output"; each test example contains only "input". '
        'The values are grids expressed as list of lists of integers (0 - 9). '
        'Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.'
    )
    return prompt

SOURCE_AGENTIC_PROMPT_LOCAL_SHA256 = hashlib.sha256(SOURCE_AGENTIC_PROMPT_TEMPLATE.encode("utf-8")).hexdigest()
SOURCE_SYSTEM_PROMPT_LOCAL_SHA256 = hashlib.sha256(SOURCE_SYSTEM_PROMPT.encode("utf-8")).hexdigest()
with open(WORK_ROOT/"p0_library_probe.json","w") as _f:
    json.dump({"available":AVAILABLE_P0_LIBRARIES,"results":P0_LIBRARY_PROBE_RESULTS},_f,indent=2)
print("P0 approved libraries available:",AVAILABLE_P0_LIBRARIES)

def render_agent_chat(messages, reasoning_effort="xhigh"):
    reasoning_effort=str(reasoning_effort)
    kwargs = dict(
        tokenize=False,
        add_generation_prompt=True,
        tools=[PYTHON_TOOL],
        enable_thinking=True,
        reasoning_effort=reasoning_effort,
        preserve_thinking=True,
    )
    try:
        rendered=tokenizer.apply_chat_template(messages, **kwargs)
        return rendered, f"qwen_tools+{reasoning_effort}+preserve_thinking"
    except TypeError as exc:
        # Preserve history semantics if this repacked tokenizer predates the explicit preserve kwarg.
        kwargs.pop("preserve_thinking", None)
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
        return rendered, f"qwen_tools+{reasoning_effort}(template-preserve-default); fallback={type(exc).__name__}"

def assert_qwen_reasoning_effort_contract():
    probe=[{"role":"system","content":"effort contract probe"},{"role":"user","content":"reply"}]
    rendered={}
    for effort in ("xhigh","medium","low"):
        rendered[effort],_=render_agent_chat(probe,effort)
    assert "Reasoning effort is set to xhigh" in rendered["xhigh"]
    assert "Reasoning effort is set to low" in rendered["low"]
    assert "Reasoning effort is set to xhigh" not in rendered["medium"]
    assert "Reasoning effort is set to low" not in rendered["medium"]
    high_rejected=False
    try:
        render_agent_chat(probe,"high")
    except Exception:
        high_rejected=True
    if not high_rejected:
        raise RuntimeError("Mounted Qwen template unexpectedly accepts reasoning_effort='high'; source contract changed.")
    return {
        "supported":["xhigh","medium","low"],
        "high_rejected":True,
        "xhigh_injects_instruction":True,
        "medium_injects_no_effort_instruction":True,
        "low_injects_instruction":True,
    }

QWEN_REASONING_EFFORT_CONTRACT=assert_qwen_reasoning_effort_contract()

# Frozen task selection: same Proxy-48; SMOKE is a deterministic subset of it.
def choose_smoke8_from_proxy():
    """v1.6: stratified to mirror Proxy-48 (tier1 32/48, tier2 12/48, tier3 4/48 -> 5/2/1), deterministic in
    Proxy-48 order. The previous first-match selector took all four tier-3 tasks and biased the feasibility gate."""
    f = proxy_source_feat.set_index("task_id").loc[PROXY48_TASK_IDS].copy()
    chosen=[]
    def add(x):
        if x is not None and x not in chosen and len(chosen)<8: chosen.append(x)
    def first(mask):
        for t in f.index[mask.to_numpy()]:
            if t not in chosen: return t
        return None
    t1=(f["proxy_tier"]==1); t2=(f["proxy_tier"]==2); t3=(f["proxy_tier"]==3)
    add(first(t3))
    add(first(t2)); add(first(t2))
    add(first(t1 & (f["proxy_easy_static"]==1)))
    add(first(t1 & (f["proxy_hard_tier1"]==1)))
    add(first(t1 & (f["proxy_n_queries"]>1)))
    add(first(t1 & (f["proxy_dynamic"]==1)))
    while len(chosen)<8 and first(t1) is not None: add(first(t1))
    for x in PROXY48_TASK_IDS:
        if len(chosen)>=8: break
        add(x)
    return chosen[:8]

if TASK_SET == "PROXY48": ACTIVE_TASK_IDS = list(PROXY48_TASK_IDS)
elif TASK_SET == "SMOKE8": ACTIVE_TASK_IDS = choose_smoke8_from_proxy()
else: raise RuntimeError(f"Unknown TASK_SET={TASK_SET}")
ACTIVE_TASK_SET=set(ACTIVE_TASK_IDS)
if len(ACTIVE_TASK_IDS)!=ACTIVE_TASK_LIMIT or len(ACTIVE_TASK_SET)!=ACTIVE_TASK_LIMIT:
    raise RuntimeError("Active task count/uniqueness mismatch")
if not ACTIVE_TASK_SET.issubset(set(PROXY48_TASK_IDS)):
    raise RuntimeError("Active set escaped frozen Proxy-48")

prompt_records=[]
for arm_name in ACT_ARM_ORDER:
    effort=ACT_ARMS[arm_name]["reasoning_effort"]
    for task_id, task in eval_challenges.items():
        user_prompt=build_agent_user_prompt(task)
        messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user_prompt}]
        rendered, method=render_agent_chat(messages,effort)
        prompt_records.append({
            "arm":arm_name,"reasoning_effort":effort,
            "task_id":task_id, "user_prompt":user_prompt, "initial_messages":messages,
            "prompt":rendered, "prompt_sha256":hashlib.sha256(rendered.encode()).hexdigest(),
            "prompt_tokens":len(tokenizer.encode(rendered, add_special_tokens=False)),
            "n_train":len(task["train"]), "n_test":len(task["test"]), "render_method":method,
        })
prompt_df=pd.DataFrame([{k:v for k,v in r.items() if k not in {"prompt","initial_messages","user_prompt"}} for r in prompt_records])
prompt_df.to_csv(WORK_ROOT/"prompt_stats_by_arm.csv", index=False)
PROMPT_RECORD_MAP_BY_ARM={
    arm:{r["task_id"]:r for r in prompt_records if r["arm"]==arm and r["task_id"] in ACTIVE_TASK_SET}
    for arm in ACT_ARM_ORDER
}
# Compatibility alias used by frozen engine warmup; choose the BUDGET/XHIGH render.
PROMPT_RECORD_MAP=PROMPT_RECORD_MAP_BY_ARM["BUDGET3072"]


chat_template=getattr(tokenizer,"chat_template","") or ""
template_sha=hashlib.sha256(chat_template.encode()).hexdigest()
if "<tool_call>" not in chat_template or "tool_response" not in chat_template:
    raise RuntimeError("Mounted Qwen chat template does not expose the required tool-call/tool-response contract.")

# Validate the *actual mounted tokenizer* on the multi-turn structure PIV will use.
_transport_probe_messages = [
    {"role":"system","content":SYSTEM_PROMPT},
    {"role":"user","content":"transport probe"},
    {"role":"assistant","content":"","reasoning_content":"PIV_REASONING_SENTINEL",
     "tool_calls":[{"type":"function","function":{"name":"python","arguments":{"code":"x=1\nx+1"}}}]},
    {"role":"tool","content":"2"},
]
_transport_rendered, _transport_method = render_agent_chat(_transport_probe_messages,"xhigh")
for _required in ("PIV_REASONING_SENTINEL", "<tool_call>", "<function=python>", "<parameter=code>", "<tool_response>", "2"):
    if _required not in _transport_rendered:
        raise RuntimeError(f"Qwen multi-turn tool-history rendering contract failed: missing {_required!r}")
if not _transport_rendered.rstrip().endswith("<think>"):
    raise RuntimeError("Qwen tool-history render did not reopen thinking for the next assistant turn.")
QWEN_TOOL_HISTORY_TRANSPORT_SHA256 = hashlib.sha256(_transport_rendered.encode("utf-8")).hexdigest()
with open(WORK_ROOT/"chat_template_fingerprint.txt","w") as f:
    f.write(f"sha256={template_sha}\ntool_schema_sha256={TOOL_SCHEMA_SHA256}\n")

profile_task_table=eval_feat[eval_feat["task_id"].isin(ACTIVE_TASK_IDS)].copy()
profile_task_table=profile_task_table.merge(proxy_source_feat,on="task_id",how="left")
_budget_prompt_map=prompt_df[prompt_df.arm=="BUDGET3072"].set_index("task_id")["prompt_tokens"]
profile_task_table["prompt_tokens_piv_initial"]=profile_task_table["task_id"].map(_budget_prompt_map)
for _arm in ACT_ARM_ORDER:
    _m=prompt_df[prompt_df.arm==_arm].set_index("task_id")["prompt_tokens"]
    profile_task_table[f"prompt_tokens_{_arm.lower()}"]=profile_task_table["task_id"].map(_m)
profile_task_table["proxy_order"]=profile_task_table["task_id"].map(proxy48_order)
profile_task_table=profile_task_table.sort_values("proxy_order").reset_index(drop=True)
profile_task_table.to_csv(WORK_ROOT/"active_task_selection.csv",index=False)

SOURCE_ALIGNMENT={
    "experiment":EXPERIMENT_ID,
    "piv_source_repo":PIV_SOURCE_REPO,
    "piv_source_commit":PIV_SOURCE_COMMIT,
    "piv_source_files":PIV_SOURCE_FILES,
    "external_sources":PIV_EXTERNAL_SOURCES,
    "proxy48_ordered_sha256":PROXY48_ORDERED_SHA256,
    "source_file_sha256":SOURCE_FILE_SHA256,
    "tool_schema_sha256":TOOL_SCHEMA_SHA256,
    "chat_template_sha256":template_sha,
    "qwen_reasoning_effort_contract":QWEN_REASONING_EFFORT_CONTRACT,
    "qwen_thinking_budget_method":"official two-generation early-stopping technique; BUDGET3072 local vLLM port",
    "qwen_tool_history_transport_sha256":QWEN_TOOL_HISTORY_TRANSPORT_SHA256,
    "source_agentic_prompt_local_sha256":SOURCE_AGENTIC_PROMPT_LOCAL_SHA256,
    "source_system_prompt_local_sha256":SOURCE_SYSTEM_PROMPT_LOCAL_SHA256,
    "source_invariants":[
        "one persistent Python REPL per lineage",
        "puzzle preloaded without test outputs",
        "tool errors returned to model",
        "self-contained solution(grid) final artifact",
        "fresh independent all-train exact verification",
        "test grids produced by verified program execution",
    ],
    "local_adaptations":[
        "direct vLLM + Qwen native tool chat template; no OpenAI Responses server",
        "lightweight subprocess worker instead of Docker/IPyBox",
        "stricter grid/domain/time/file-system safety",
        "bounded 6-execution P0 tool budget and context guard",
        "source-style max_retries=2 reliability retries share the same total token/tool/turn budgets",
         "three-arm activation ablation: LOW, MEDIUM, BUDGET3072 under the same 12288 generated-token/task cap",
        "BUDGET3072 follows Qwen's documented two-generation thinking-budget pattern; injected early-stop tokens logged separately",
    ],
}
with open(WORK_ROOT/"SOURCE_ALIGNMENT.json","w") as f: json.dump(SOURCE_ALIGNMENT,f,indent=2)
print("Chat template/tool contract PASS | SHA:",template_sha)
print("ACTIVE TASKS:",len(ACTIVE_TASK_IDS),TASK_SET)
display(prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].describe().to_frame("initial_prompt_tokens").round(1))


In [ ]:
# =====================================================================================
# CELL 5 — PIV CONTEXT / COMPUTE SAFETY (preserve history; do not add P3 compaction in P0)
# =====================================================================================
active_prompt_tokens=prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].to_numpy()
max_initial_prompt=int(active_prompt_tokens.max())
CONTEXT_SAFETY_TOKENS=512
MAX_RENDERED_INPUT_TOKENS=MAX_MODEL_LEN-PER_TURN_MAX_TOKENS-CONTEXT_SAFETY_TOKENS
if max_initial_prompt > MAX_RENDERED_INPUT_TOKENS:
    raise RuntimeError(f"Initial PIV prompt exceeds context: {max_initial_prompt} > {MAX_RENDERED_INPUT_TOKENS}")
MAX_TOOL_OUTPUT_CHARS=4000
MAX_TOOL_RAW_ARTIFACT_CHARS=100000
MAX_CODE_CHARS=30000
print("PIV CONTEXT CONFIG")
print("  max_model_len           :",MAX_MODEL_LEN)
print("  max initial prompt      :",max_initial_prompt)
print("  per-turn max generation:",PER_TURN_MAX_TOKENS)
print("  total generated cap     :",TOTAL_GENERATED_TOKEN_CAP)
print("  max rendered input      :",MAX_RENDERED_INPUT_TOKENS)
print("  max tool output to model:",MAX_TOOL_OUTPUT_CHARS,"chars")
print("  history policy          : PRESERVE (P0); no P3 compact-state intervention")


In [ ]:
# =====================================================================================
# CELL 6 — QWEN TOOL XML PARSER + PERSISTENT LOCAL REPL + FRESH EXACT VERIFIER + SELF-TESTS
# =====================================================================================
import ast as py_ast
import re, subprocess, selectors, tempfile, uuid

DANGEROUS_IMPORT_ROOTS={"os","subprocess","pathlib","shutil","socket","requests","urllib","http","ftplib","paramiko","builtins","importlib","inspect"}
DANGEROUS_CALL_NAMES={"open","exec","eval","compile","__import__","input","breakpoint","globals","locals","vars","getattr","setattr","delattr","exit","quit"}
DANGEROUS_DUNDER_ATTRS={"__subclasses__","__globals__","__builtins__","__code__","__closure__","__getattribute__"}
ALLOWED_SYS_ATTRS={"setrecursionlimit","getrecursionlimit","maxsize"}

def canonical_grid(grid):
    if isinstance(grid,np.ndarray): grid=grid.tolist()
    if not is_rectangular_grid(grid): return None
    return tuple(tuple(int(x) for x in row) for row in grid)
def grids_equal(a,b):
    ca,cb=canonical_grid(a),canonical_grid(b)
    return ca is not None and cb is not None and ca==cb

def code_defines_solution(code):
    try: tree=py_ast.parse(code)
    except Exception: return False
    return any(isinstance(n,(py_ast.FunctionDef,py_ast.AsyncFunctionDef)) and n.name=="solution" for n in tree.body)

def extract_last_solution_code(text):
    blocks=re.findall(r"```(?:python|py)?\s*\n?(.*?)```", text or "", flags=re.I|re.S)
    for code in reversed(blocks):
        code=code.strip()
        if code_defines_solution(code): return code
    return None

def split_qwen_reasoning(text, action_pos=None):
    text=text or ""
    prefix=text if action_pos is None else text[:action_pos]
    if "</think>" in prefix:
        a,b=prefix.rsplit("</think>",1)
        return a.replace("<think>","",1).strip(), b.strip(), "think_close"
    # Generation prompt already opens <think>; without a close, pre-action text is reasoning.
    return prefix.replace("<think>","",1).strip(), "", "implicit_open_think"

def _action_region(text):
    """Prefer the model-action region after the final </think>; fall back to all text if no close exists."""
    text=text or ""
    if "</think>" in text:
        pos=text.rfind("</think>")+len("</think>")
        return text[pos:],pos
    return text,0

def parse_tool_calls(text):
    text=text or ""
    region,offset=_action_region(text)
    calls=[]
    pattern=r"<tool_call>\s*<function=([^>\s]+)>\s*(.*?)</function>\s*</tool_call>"
    for m in re.finditer(pattern,region,re.S):
        name=m.group(1).strip(); body=m.group(2); args={}
        for p in re.finditer(r"<parameter=([^>\s]+)>\s*(.*?)\s*</parameter>",body,re.S):
            val=p.group(2)
            # v1.6: unwrap a fenced ```python payload; the fence itself is not Python.
            _fm=re.fullmatch(r"```(?:python|py)?[ \t]*\n?(.*?)\n?```",val.strip(),flags=re.S|re.I)
            if _fm: val=_fm.group(1)
            args[p.group(1).strip()]=val
        calls.append({"name":name,"arguments":args,"start":offset+m.start(),"end":offset+m.end(),"raw":m.group(0)})
    return calls

def parse_agent_turn(text):
    text=text or ""
    tcs=parse_tool_calls(text)
    if tcs:
        reasoning,content,rmode=split_qwen_reasoning(text,tcs[0]["start"])
        suffix=text[tcs[-1]["end"]:].strip()
        suffix=re.sub(r"(?:<\|im_end\|>|<\|endoftext\|>)\s*$","",suffix).strip()
        return {"kind":"tool","tool_calls":tcs,"tool_name":tcs[0]["name"],"tool_args":tcs[0]["arguments"],
                "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
                "suffix_after_tool":suffix,"final_code":None}
    action_text,_=_action_region(text)
    if "<tool_call>" in action_text:   # v1.6: prose merely mentioning <function= is not a malformed call
        reasoning,content,rmode=split_qwen_reasoning(text,None)
        return {"kind":"malformed_tool","tool_calls":[],"tool_name":None,"tool_args":{},
                "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
                "suffix_after_tool":"","final_code":None}
    code=extract_last_solution_code(action_text)
    reasoning,content,rmode=split_qwen_reasoning(text,None)
    return {"kind":"final" if code else "text","tool_calls":[],"tool_name":None,"tool_args":{},
            "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
            "suffix_after_tool":"","final_code":code}

def generation_finish_reason_ok(finish_reason):
    # Mirrors the source client's rule that a truncated response is not a valid turn result.
    return str(finish_reason).lower() == "stop"

# v1.6: tool output is inserted verbatim into the next prompt. Neutralize chat-control strings the model's
# own code could print so they cannot open/close turns or tool blocks.
_CHAT_CONTROL_PATTERNS=[("<|","< |"),("<tool_response>","< tool_response>"),("</tool_response>","< /tool_response>"),
                        ("<tool_call>","< tool_call>"),("</tool_call>","< /tool_call>"),
                        ("<think>","< think>"),("</think>","< /think>")]
def sanitize_tool_output(text):
    text=str(text or "")
    for a,b in _CHAT_CONTROL_PATTERNS: text=text.replace(a,b)
    return text

def salvage_final_code_on_length(text, parsed):
    """v1.6 LOCAL: a length-capped final turn whose action region (after </think>) already contains a CLOSED,
    ast-parseable ```python block defining solution() is demonstrably complete; retrying it would only burn
    budget. Drafts inside an unclosed <think> are never salvaged."""
    if parsed.get("kind")!="final" or parsed.get("reasoning_mode")!="think_close": return None
    return parsed.get("final_code")

def guard_python_code(code, *, final=False):
    if not isinstance(code,str) or not code.strip(): return False,"empty_code"
    if len(code)>MAX_CODE_CHARS: return False,"code_too_long"
    try: tree=py_ast.parse(code)
    except SyntaxError as e: return False,f"syntax_error:{e.msg}"

    # `sys.setrecursionlimit(...)` is common in DFS/flood-fill ARC solutions.
    # Permit only a narrow sys surface; block module-table/process introspection.
    sys_aliases=set()
    for _n in py_ast.walk(tree):
        if isinstance(_n,py_ast.Import):
            for _a in _n.names:
                if _a.name=="sys": sys_aliases.add(_a.asname or "sys")
        elif isinstance(_n,py_ast.ImportFrom) and (_n.module or "")=="sys":
            imported={_a.name for _a in _n.names}
            if not imported.issubset(ALLOWED_SYS_ATTRS):
                return False,f"blocked_sys_importfrom:{sorted(imported)}"

    for node in py_ast.walk(tree):
        if isinstance(node,py_ast.Name) and node.id=="__builtins__":
            return False,"blocked_builtins_escape"
        if isinstance(node,py_ast.Attribute) and node.attr in DANGEROUS_DUNDER_ATTRS:
            return False,f"blocked_dunder_attribute:{node.attr}"
        if isinstance(node,py_ast.Attribute) and isinstance(node.value,py_ast.Name) and node.value.id in sys_aliases:
            if node.attr not in ALLOWED_SYS_ATTRS:
                return False,f"blocked_sys_attribute:{node.attr}"
        if isinstance(node,py_ast.Constant) and isinstance(node.value,str) and "/kaggle" in node.value.lower():
            return False,"blocked_host_path_literal"
        if isinstance(node,(py_ast.Import,py_ast.ImportFrom)):
            names=[a.name.split('.')[0] for a in node.names] if isinstance(node,py_ast.Import) else [(node.module or '').split('.')[0]]
            if any(x in DANGEROUS_IMPORT_ROOTS for x in names): return False,f"blocked_import:{names}"
        if isinstance(node,py_ast.Call):
            if isinstance(node.func,py_ast.Name) and node.func.id in DANGEROUS_CALL_NAMES:
                return False,f"blocked_call:{node.func.id}"
            if isinstance(node.func,py_ast.Attribute) and node.func.attr in DANGEROUS_DUNDER_ATTRS:
                return False,f"blocked_dunder_call:{node.func.attr}"
        if isinstance(node,py_ast.Raise):
            exc=node.exc
            if isinstance(exc,py_ast.Name) and exc.id=="SystemExit":
                return False,"blocked_system_exit"
            if isinstance(exc,py_ast.Call) and isinstance(exc.func,py_ast.Name) and exc.func.id=="SystemExit":
                return False,"blocked_system_exit"
        if isinstance(node,(py_ast.Assign,py_ast.AnnAssign,py_ast.AugAssign,py_ast.NamedExpr)):
            targets=[]
            if isinstance(node,py_ast.Assign): targets=node.targets
            else: targets=[node.target]
            def rooted_at_puzzle(t):
                while isinstance(t,(py_ast.Subscript,py_ast.Attribute)):
                    t=t.value
                return isinstance(t,py_ast.Name) and t.id=="puzzle"
            for t in targets:
                if rooted_at_puzzle(t): return False,"protected_puzzle_assignment"
        if isinstance(node,py_ast.Call) and isinstance(node.func,py_ast.Attribute):
            base=node.func.value
            while isinstance(base,(py_ast.Subscript,py_ast.Attribute)): base=base.value
            if isinstance(base,py_ast.Name) and base.id=="puzzle" and node.func.attr in {"update","clear","pop","popitem","setdefault","append","extend","insert","remove"}:
                return False,"protected_puzzle_mutation"
        if isinstance(node,py_ast.Expr) and isinstance(node.value,py_ast.Name) and node.value.id=="puzzle":
            return False,"blocked_whole_puzzle_display"
        if isinstance(node,py_ast.Call) and isinstance(node.func,py_ast.Name) and node.func.id=="print":
            if any(isinstance(a,py_ast.Name) and a.id=="puzzle" for a in node.args):
                return False,"blocked_whole_puzzle_print"
    if final and not code_defines_solution(code): return False,"missing_solution_function"
    return True,"ok"

def compact_tool_output(text,limit=MAX_TOOL_OUTPUT_CHARS):
    text=str(text or "")
    if len(text)<=limit: return text
    keep=max(1,(limit-120)//2)
    return text[:keep]+f"\n...[tool output truncated; original chars={len(text)}]...\n"+text[-keep:]

WORKER_SCRIPT = r"""
import sys,json,traceback,io,contextlib,copy,ast,os
def _piv_audit(event,args):
    try:
        if event=="open" and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV sandbox blocks host /kaggle filesystem access")
        if event in {"os.listdir","os.scandir","os.chdir","os.remove","os.rename","os.rmdir","os.mkdir"} and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV sandbox blocks host /kaggle filesystem access")
        if event=="subprocess.Popen" or event=="os.system" or event.startswith("socket."):
            raise PermissionError("PIV sandbox blocks process/network access")
    except PermissionError:
        raise
    except Exception:
        pass
sys.addaudithook(_piv_audit)
ns={"__name__":"__main__"}
original_puzzle=None
for line in sys.stdin:
    try:
        msg=json.loads(line); op=msg.get("op")
        if op=="init":
            original_puzzle=copy.deepcopy(msg["puzzle"])
            ns={"__name__":"__main__","puzzle":copy.deepcopy(original_puzzle)}
            out={"ok":True,"text":"REPL initialized."}
        elif op=="exec":
            # Restore the protected puzzle binding before every cell. Other REPL state persists.
            ns["puzzle"]=copy.deepcopy(original_puzzle)
            buf=io.StringIO()
            try:
                with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                    # Jupyter-like semantics: execute statements, then display repr(last expression).
                    tree=ast.parse(msg["code"],mode="exec")
                    if tree.body and isinstance(tree.body[-1],ast.Expr):
                        prefix=ast.Module(body=tree.body[:-1],type_ignores=[])
                        if prefix.body: exec(compile(prefix,"<piv-repl>","exec"),ns,ns)
                        value=eval(compile(ast.Expression(tree.body[-1].value),"<piv-repl>","eval"),ns,ns)
                        if value is not None: print(repr(value))
                    else:
                        exec(compile(tree,"<piv-repl>","exec"),ns,ns)
                out={"ok":True,"text":buf.getvalue() or "Code executed successfully. No output was returned."}
            except BaseException:
                out={"ok":False,"text":buf.getvalue()+traceback.format_exc(limit=8)}
        elif op=="close":
            print(json.dumps({"ok":True,"text":"closing"}),flush=True); break
        else: out={"ok":False,"text":"Unknown worker op"}
    except BaseException:
        out={"ok":False,"text":traceback.format_exc(limit=8)}
    print(json.dumps(out,ensure_ascii=False),flush=True)
"""

class PersistentPythonWorker:
    def __init__(self,puzzle,timeout_sec=10.0):
        self.puzzle=puzzle; self.timeout_sec=float(timeout_sec); self.proc=None; self.successful_cells=[]; self.restart_count=0
        self.start()
    def start(self):
        self.close(silent=True)
        self.proc=subprocess.Popen([sys.executable,"-u","-c",WORKER_SCRIPT],stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                                   stderr=subprocess.DEVNULL,text=True,bufsize=1,cwd=tempfile.mkdtemp(prefix="piv_repl_"))
        r=self._request({"op":"init","puzzle":self.puzzle},timeout=max(2.0,self.timeout_sec))
        if not r.get("ok"): raise RuntimeError(f"REPL init failed: {r}")
        # Replay only previously successful bounded cells after a timeout/crash.
        if self.successful_cells:
            cells=list(self.successful_cells); self.successful_cells=[]
            for c in cells:
                rr=self.execute(c,record=True,replay=True)
                if not rr.get("ok"): break
    def _request(self,msg,timeout):
        if self.proc is None or self.proc.poll() is not None: raise RuntimeError("REPL process is not alive")
        self.proc.stdin.write(json.dumps(msg,ensure_ascii=False)+"\n"); self.proc.stdin.flush()
        sel=selectors.DefaultSelector(); sel.register(self.proc.stdout,selectors.EVENT_READ)
        events=sel.select(timeout); sel.close()
        if not events: raise TimeoutError(f"REPL tool exceeded {timeout}s")
        line=self.proc.stdout.readline()
        if not line: raise RuntimeError("REPL worker closed stdout")
        return json.loads(line)
    def execute(self,code,record=True,replay=False):
        ok,reason=guard_python_code(code,final=False)
        if not ok: return {"ok":False,"text":f"Tool code rejected by local safety guard: {reason}","guard_rejected":True,"restarted":False}
        try:
            r=self._request({"op":"exec","code":code},self.timeout_sec); r["restarted"]=False
            if r.get("ok") and record: self.successful_cells.append(code)
            return r
        except TimeoutError:
            self.restart_count+=1
            old=list(self.successful_cells)
            self.successful_cells=old
            try: self.start()
            except Exception: pass
            return {"ok":False,"text":f"Code execution terminated after {self.timeout_sec}s. Persistent worker was restarted; prior successful cells were replayed when possible.","timeout":True,"restarted":True}
        except Exception as e:
            self.restart_count+=1
            restarted=False
            try:
                self.start(); restarted=True
            except Exception:
                restarted=False
            return {"ok":False,"text":f"REPL infrastructure error: {type(e).__name__}: {e}. "
                                      f"Worker restart attempted; prior successful cells replayed when possible.",
                    "infrastructure_error":True,"restarted":restarted}
    def close(self,silent=False):
        p=getattr(self,"proc",None)
        if p is not None and p.poll() is None:
            try: p.terminate(); p.wait(timeout=1)
            except Exception:
                try: p.kill()
                except Exception: pass
        self.proc=None

VERIFIER_SCRIPT = r"""
import sys,json,copy,traceback,numbers,os
payload=json.loads(sys.stdin.read()); code=payload["code"]; task=payload["task"]
def _piv_audit(event,args):
    try:
        if event=="open" and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV verifier blocks host /kaggle filesystem access")
        if event in {"os.listdir","os.scandir","os.chdir","os.remove","os.rename","os.rmdir","os.mkdir"} and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV verifier blocks host /kaggle filesystem access")
        if event=="subprocess.Popen" or event=="os.system" or event.startswith("socket."):
            raise PermissionError("PIV verifier blocks process/network access")
    except PermissionError:
        raise
    except Exception:
        pass
sys.addaudithook(_piv_audit)
ns={"__name__":"__main__"}
def clean(g):
    try:
        import numpy as np
        if isinstance(g,np.ndarray): g=g.tolist()
    except Exception:
        pass
    # Source clean_grid accepts iterable rows. Normalize common iterable forms before LOCAL shape/domain checks.
    try:
        rows=list(g)
    except Exception:
        raise ValueError("invalid_grid_rows")
    if not (1<=len(rows)<=30):
        raise ValueError("invalid_grid_rows")
    out=[]
    for r in rows:
        try:
            if hasattr(r,"tolist"):
                r=r.tolist()
            rr0=list(r)
        except Exception:
            raise ValueError("invalid_grid_rows")
        out.append(rr0)
    widths=[len(r) for r in out]
    if not widths or min(widths)<1 or max(widths)>30 or len(set(widths))!=1:
        raise ValueError("invalid_grid_shape")
    clean_out=[]
    for r in out:
        rr=[]
        for x in r:
            if isinstance(x,bool) or not isinstance(x,numbers.Integral): raise ValueError("non_integer_cell")
            x=int(x)
            if x<0 or x>9: raise ValueError("cell_out_of_domain")
            rr.append(x)
        clean_out.append(rr)
    return clean_out

RESULT_SENTINEL="PIV_RESULT:"
result={"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],
        "test_execution_ok":False,"any_test_output_ok":False,"test_errors":[],
        "train_error":None,"test_error":None}
def emit_result():
    # Leading newline keeps the machine-readable result separate even if model code used print(..., end="").
    sys.__stdout__.write("\n"+RESULT_SENTINEL+json.dumps(result)+"\n")
    sys.__stdout__.flush()
try:
    exec(code,ns,ns); fn=ns.get("solution")
    if not callable(fn): raise ValueError("solution_not_callable")
except BaseException:
    result["train_error"]=traceback.format_exc(limit=10)
    emit_result(); raise SystemExit(0)

# SOURCE-FAITHFUL ordering: establish the train gate first.
for i,ex in enumerate(task["train"]):
    try:
        pred=clean(fn(copy.deepcopy(ex["input"])))
        exact=(pred==ex["output"])
        result["train"].append({"index":i,"exact":exact,"pred":pred if not exact else None,"error":None})
    except BaseException:
        result["train"].append({"index":i,"exact":False,"pred":None,"error":traceback.format_exc(limit=8)})
result["all_train_exact"]=bool(result["train"]) and all(x["exact"] for x in result["train"])

# Test execution is a separate per-output state, matching source produce_output_grid.py.
# One failing test input must not erase successful outputs for other test inputs.
if result["all_train_exact"]:
    tests=[]; test_errors=[]
    for j,ex in enumerate(task["test"]):
        try:
            tests.append(clean(fn(copy.deepcopy(ex["input"]))))
            test_errors.append(None)
        except BaseException:
            tests.append(None)
            test_errors.append(traceback.format_exc(limit=10))
    result["test_outputs"]=tests
    result["test_errors"]=test_errors
    result["test_execution_ok"]=all(x is not None for x in tests)
    result["any_test_output_ok"]=any(x is not None for x in tests)
    result["ok"]=result["any_test_output_ok"]
    if not result["test_execution_ok"]:
        result["test_error"]="\n".join(e for e in test_errors if e)

emit_result()
"""

def task_without_test_solutions(task):
    return {"train":copy.deepcopy(task["train"]),"test":[{"input":copy.deepcopy(x["input"])} for x in task["test"]]}

def fresh_verify_solution(code,task,timeout_sec=VERIFIER_TIMEOUT_SEC):
    ok,reason=guard_python_code(code,final=True)
    if not ok: return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":reason,"verifier_status":"guard_rejected"}
    payload={"code":code,"task":task_without_test_solutions(task)}
    try:
        cp=subprocess.run([sys.executable,"-u","-c",VERIFIER_SCRIPT],input=json.dumps(payload),text=True,capture_output=True,
                          timeout=float(timeout_sec),cwd=tempfile.mkdtemp(prefix="piv_verify_"))
        if cp.returncode!=0: return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":cp.stderr[-4000:],"verifier_status":"process_error"}
        sentinel="PIV_RESULT:"
        payload_lines=[ln[len(sentinel):] for ln in cp.stdout.splitlines() if ln.startswith(sentinel)]
        if not payload_lines:
            return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],
                    "error":"missing_verifier_result_sentinel","verifier_status":"protocol_error",
                    "stdout_tail":cp.stdout[-2000:]}
        r=json.loads(payload_lines[-1])
        if r.get("all_train_exact") and r.get("test_execution_ok"):
            r["verifier_status"]="train_exact_test_ok"
        elif r.get("all_train_exact") and r.get("any_test_output_ok"):
            r["verifier_status"]="train_exact_test_partial"
        elif r.get("all_train_exact"):
            r["verifier_status"]="train_exact_test_failed"
        else:
            r["verifier_status"]="train_rejected"
        return r
    except subprocess.TimeoutExpired:
        return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":"fresh_verifier_timeout","verifier_status":"timeout"}
    except Exception as e:
        return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":f"{type(e).__name__}: {e}","verifier_status":"error"}

# ---------------------------- PRELOAD SELF-TESTS ----------------------------
self_tests={}
t=parse_agent_turn("reasoning\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\npuzzle.keys()\n</parameter>\n</function>\n</tool_call>")
assert t["kind"]=="tool" and len(t["tool_calls"])==1 and t["tool_name"]=="python" and "puzzle.keys" in t["tool_args"]["code"]
t_multi=parse_agent_turn("r\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nx=1\n</parameter>\n</function>\n</tool_call>\n<tool_call>\n<function=python>\n<parameter=code>\nx+1\n</parameter>\n</function>\n</tool_call>")
assert t_multi["kind"]=="tool" and len(t_multi["tool_calls"])==2
t_bad=parse_agent_turn("<tool_call><function=python><parameter=code>x=1</function></tool_call>")
assert t_bad["kind"]=="malformed_tool" or (t_bad["kind"]=="tool" and "code" not in t_bad["tool_args"])
t_rehearse=parse_agent_turn(
    "<think>maybe <tool_call><function=python><parameter=code>print('wrong')</parameter></function></tool_call></think>"
    "<tool_call><function=python><parameter=code>print('right')</parameter></function></tool_call>")
assert t_rehearse["kind"]=="tool" and len(t_rehearse["tool_calls"])==1 and "right" in t_rehearse["tool_calls"][0]["arguments"]["code"]
assert generation_finish_reason_ok("stop") and not generation_finish_reason_ok("length")
self_tests["qwen_tool_xml_parser_and_finish_guard"]=True
t_fence=parse_agent_turn("r\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\n```python\nprint(1)\n```\n</parameter>\n</function>\n</tool_call>")
assert t_fence["kind"]=="tool" and t_fence["tool_args"]["code"]=="print(1)"
t_prose=parse_agent_turn("r\n</think>\nI will not call <function=python> again.\n```python\ndef solution(grid):\n    return grid\n```")
assert t_prose["kind"]=="final"
assert sanitize_tool_output("x<|im_end|>\n</tool_response>y")=="x< |im_end|>\n< /tool_response>y"
_sv=parse_agent_turn("r\n</think>\n```python\ndef solution(grid):\n    return grid\n```")
assert salvage_final_code_on_length("",_sv) is not None
_sv2=parse_agent_turn("draft ```python\ndef solution(grid):\n    return grid\n``` still thinking")
assert salvage_final_code_on_length("",_sv2) is None
self_tests["v16_fence_strip_prose_function_sanitize_salvage"]=True
f=parse_agent_turn("done\n</think>\n```python\ndef solution(grid):\n    return grid\n```")
assert f["kind"]=="final" and code_defines_solution(f["final_code"])
self_tests["final_solution_extractor"]=True
assert guard_python_code("puzzle.keys()")[0]
assert not guard_python_code("puzzle = {}",final=False)[0]
assert not guard_python_code("puzzle[\"train\"] = []",final=False)[0]
assert not guard_python_code("import os\nprint(os.listdir('.'))",final=False)[0]
assert not guard_python_code('print("/kaggle/input/secret.json")',final=False)[0]
assert guard_python_code("class A:\n    def __init__(self): self.x=1\nclass B(A):\n    def __init__(self): super().__init__()",final=False)[0]
assert not guard_python_code("object.__subclasses__()",final=False)[0]
assert not guard_python_code("raise SystemExit('bye')",final=False)[0]
assert guard_python_code("import sys\nsys.setrecursionlimit(10000)",final=False)[0]
assert guard_python_code("import sys as s\ns.setrecursionlimit(10000)",final=False)[0]
assert not guard_python_code("import sys\nsys.modules",final=False)[0]
self_tests["tool_safety_guard"]=True
_fake={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]}]}
w=PersistentPythonWorker(task_without_test_solutions(_fake),timeout_sec=5.0)
r0=w.execute("puzzle.keys()"); r1=w.execute("x=41"); r2=w.execute("x+1")
assert r0["ok"] and "dict_keys" in r0["text"] and r1["ok"] and r2["ok"] and "42" in r2["text"]
ra=w.execute("g=puzzle['train'][0]['input']; g[0][0]=9")
rb=w.execute("puzzle['train'][0]['input'][0][0]")
assert rb["ok"] and "1" in rb["text"]
rc=w.execute("raise SystemExit('bye')")
rd=w.execute("1+1")
assert (not rc["ok"]) and rd["ok"] and "2" in rd["text"]
w.close()
self_tests["persistent_repl_state_and_expression_display"]=True
v=fresh_verify_solution("def solution(grid):\n    return [row[:] for row in grid]",_fake,timeout_sec=5)
assert v["all_train_exact"] and v["test_outputs"]==[[[2]]]
v_noise=fresh_verify_solution(
    "print('noise', end='')\ndef solution(grid):\n    return [row[:] for row in grid]",
    _fake,timeout_sec=5)
assert v_noise["all_train_exact"] and v_noise["test_outputs"]==[[[2]]]
self_tests["verifier_result_sentinel"]=True
self_tests["fresh_exact_verifier"]=True
v2=fresh_verify_solution("def solution(grid):\n    return [[9]]",_fake,timeout_sec=5)
assert not v2["all_train_exact"] and v2["test_outputs"]==[]
self_tests["train_fail_blocks_test_execution"]=True
v3=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]: raise RuntimeError('test-only failure')\n    return tuple(tuple(x for x in row) for row in grid)",
    _fake,timeout_sec=5)
assert v3["all_train_exact"] and not v3["test_execution_ok"] and v3["test_outputs"]==[None]
self_tests["train_exact_survives_test_execution_failure"]=True
_fake_multi={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]},{"input":[[3]]}]}
v4=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]: raise RuntimeError('one test fails')\n    return tuple(tuple(x for x in row) for row in grid)",
    _fake_multi,timeout_sec=5)
assert v4["all_train_exact"] and v4["test_outputs"][0] is None and v4["test_outputs"][1]==[[3]]
self_tests["partial_multiquery_test_outputs_preserved"]=True
with open(WORK_ROOT/"preload_self_tests.json","w") as f: json.dump(self_tests,f,indent=2)
print("PIV preload self-tests PASS:",self_tests)


In [ ]:
# =====================================================================================
# CELL 7 — CPU-SAFE PREFLIGHT OR REAL vLLM/TP=4 MODEL LOAD (Q1-FROZEN ENGINE)
# Dependency versions were already pinned in Cell 1 for PREFLIGHT/production parity.
# =====================================================================================
gpu_info = []
notebook_start = NOTEBOOK_WALL_START
model_load_sec = 0.0
warmup_sec = 0.0
arc_llm = None

RUNTIME_VLLM_VERSION = pkg_version("vllm")
RUNTIME_TORCH_VERSION = pkg_version("torch")
RUNTIME_TRANSFORMERS_VERSION = pkg_version("transformers")
RUNTIME_TOKENIZERS_VERSION = pkg_version("tokenizers")
RUNTIME_HF_HUB_VERSION = pkg_version("huggingface-hub")
RUNTIME_CUDA_VERSION = None

deadline = notebook_start + TOTAL_RUNTIME_SEC - SAFETY_RESERVE_SEC

# Fingerprint the mounted checkpoint without hashing the huge shard bytes.
with open(index_path, "r", encoding="utf-8") as f:
    _model_index = json.load(f)
_model_shards = sorted(set((_model_index.get("weight_map") or {}).values()))
_model_shard_layout = []
for _name in _model_shards:
    _p = MODEL_PATH / _name
    _model_shard_layout.append({
        "name": _name,
        "bytes": int(_p.stat().st_size) if _p.exists() else None,
    })

MODEL_ARTIFACT_FINGERPRINT_PAYLOAD = {
    "model_path": str(MODEL_PATH),
    "config_sha256": small_file_sha256(config_path),
    "safetensors_index_sha256": small_file_sha256(index_path),
    "shard_layout": _model_shard_layout,
    "chat_template_sha256": template_sha,
    "tokenizer_class": tokenizer.__class__.__name__,
    "tokenizer_vocab_size": len(tokenizer),
}
MODEL_ARTIFACT_FINGERPRINT_SHA256 = stable_json_sha256(MODEL_ARTIFACT_FINGERPRINT_PAYLOAD)

ENGINE_CONFIG = {
    "model": str(MODEL_PATH),
    "tensor_parallel_size": 4,
    "distributed_executor_backend": "mp",
    "dtype": "auto",
    "load_format": "safetensors",
    "trust_remote_code": False,
    "language_model_only": True,
    "max_model_len": MAX_MODEL_LEN,
    "max_num_seqs": MAX_NUM_SEQS,
    "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "enable_prefix_caching": bool(ENABLE_PREFIX_CACHING),
    "seed": SEED,
}
ENGINE_CONFIG_SHA256 = stable_json_sha256(ENGINE_CONFIG)

if not USE_MODEL:
    # True no-allocation path: package parity is real, but do not import vLLM/CUDA runtime.
    SamplingParams = None
    print("=" * 116)
    print("PREFLIGHT CPU-SAFE / DEPENDENCY-PARITY MODE")
    print("=" * 116)
    print("vLLM package metadata :", RUNTIME_VLLM_VERSION)
    print("Transformers          :", RUNTIME_TRANSFORMERS_VERSION)
    print("Tokenizers            :", RUNTIME_TOKENIZERS_VERSION)
    print("HF Hub                :", RUNTIME_HF_HUB_VERSION)
    print("Skipping vLLM import, CUDA checks, SamplingParams API check and 27B allocation.")
    print("GPU/runtime-specific paths remain covered by ACT_SMOKE_GPU.")
else:
    import torch
    import vllm
    from vllm import LLM, SamplingParams
    from vllm.inputs import TokensPrompt

    RUNTIME_VLLM_VERSION = vllm.__version__
    RUNTIME_TORCH_VERSION = torch.__version__
    RUNTIME_CUDA_VERSION = torch.version.cuda

    print("=" * 116)
    print("REAL vLLM / CUDA PREFLIGHT")
    print("=" * 116)
    print("PyTorch :", RUNTIME_TORCH_VERSION)
    print("CUDA    :", RUNTIME_CUDA_VERSION)
    print("vLLM    :", RUNTIME_VLLM_VERSION)
    print("Transformers:", pkg_version("transformers"))

    if RUNTIME_VLLM_VERSION != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        raise RuntimeError("vLLM runtime drift after import.")
    if pkg_version("transformers") != EXPECTED_RUNTIME_PACKAGES["transformers"]:
        raise RuntimeError("Transformers runtime drift after vLLM setup.")

    # Fail fast on the exact SamplingParams fields used later, BEFORE allocating 27B.
    _sampling_api_probe = SamplingParams(
        temperature=1.0,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        presence_penalty=0.0,
        repetition_penalty=1.0,
        max_tokens=16,
        ignore_eos=False,
        seed=SEED,
        skip_special_tokens=False,
        spaces_between_special_tokens=False,
    )
    print("SamplingParams API self-test: PASS")

    print("\nLOADING QWEN3.8-27B-FP8 ON 4×L4")
    print("GPUs visible:", torch.cuda.device_count())

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable. Select Kaggle 4×L4 accelerator.")
    if torch.cuda.device_count() != 4:
        raise RuntimeError(f"Expected 4 GPUs; PyTorch sees {torch.cuda.device_count()}.")

    for i in range(4):
        cc = torch.cuda.get_device_capability(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
        name = torch.cuda.get_device_name(i)
        gpu_info.append({"gpu": i, "name": name, "memory_gib": mem, "cc": f"{cc[0]}.{cc[1]}"})
        print(f"GPU {i}: {name} | {mem:.2f} GiB | CC {cc[0]}.{cc[1]}")

    load_start = time.perf_counter()
    arc_llm = LLM(**ENGINE_CONFIG)
    model_load_sec = time.perf_counter() - load_start
    print(f"MODEL LOAD COMPLETE: {model_load_sec:.1f} sec")

    # Real generation smoke request catches engine/scheduler/model problems immediately.
    _active_prompts = [r for r in prompt_records if r["task_id"] in ACTIVE_TASK_SET]
    _med = float(np.median([r["prompt_tokens"] for r in _active_prompts]))
    probe_record = min(_active_prompts, key=lambda r: abs(r["prompt_tokens"] - _med))
    warm_params = SamplingParams(
        temperature=0.0,
        max_tokens=32,
        ignore_eos=False,
        skip_special_tokens=False,
    )
    t0 = time.perf_counter()
    _probe_ids=tokenizer.encode(probe_record["prompt"],add_special_tokens=False)
    # Validate both text prompts and token-exact continuation prompts in one real vLLM wave.
    warm = arc_llm.generate(
        [probe_record["prompt"], TokensPrompt(prompt_token_ids=_probe_ids)],
        [warm_params, warm_params],
        use_tqdm=False,
    )
    warmup_sec = time.perf_counter() - t0
    assert len(warm) == 2 and all(len(x.outputs) >= 1 for x in warm)
    print(f"REAL MODEL WARMUP (text + token-prompt continuation API): PASS ({warmup_sec:.2f} sec)")

RUNTIME_STACK = {
    "vllm": pkg_version("vllm"),
    "torch": pkg_version("torch"),
    "transformers": pkg_version("transformers"),
    "tokenizers": pkg_version("tokenizers"),
    "huggingface-hub": pkg_version("huggingface-hub"),
    "cuda": RUNTIME_CUDA_VERSION,
}
RUNTIME_STACK_SHA256 = stable_json_sha256(RUNTIME_STACK)

print("Model artifact fingerprint:", MODEL_ARTIFACT_FINGERPRINT_SHA256)
print("Engine config fingerprint :", ENGINE_CONFIG_SHA256)
print("Runtime stack fingerprint :", RUNTIME_STACK_SHA256)
print("Safety reserve:", SAFETY_RESERVE_SEC / 60, "minutes")


In [ ]:
# =====================================================================================
# CELL 8 — ACT-v1.8 THREE-ARM STATEFUL PIV LOOP + FULL EFFECT INSTRUMENTATION
# =====================================================================================
import uuid

QWEN_OFFICIAL_SAMPLING_REFERENCE={
    "thinking":{"temperature":1.0,"top_p":0.95,"top_k":20,"min_p":0.0,"presence_penalty":0.0,"repetition_penalty":1.0}
}
BASE_SAMPLING={
    "temperature":1.0,"top_p":0.95,"top_k":20,"min_p":0.0,
    "presence_penalty":0.0,"repetition_penalty":1.0,"seed":ACT_BASE_SEED,
}
AGENT_BUDGET={
    "agent_batch_size":AGENT_BATCH_SIZE,"max_tool_calls":MAX_TOOL_CALLS,"max_model_turns":MAX_MODEL_TURNS,
    "max_retries":MAX_RETRIES,"per_turn_max_tokens":PER_TURN_MAX_TOKENS,
    "total_generated_token_cap":TOTAL_GENERATED_TOKEN_CAP,
    "tool_timeout_sec":TOOL_TIMEOUT_SEC,"verifier_timeout_sec":VERIFIER_TIMEOUT_SEC,
    "max_tool_output_chars":MAX_TOOL_OUTPUT_CHARS,"history_policy":"preserve_thinking_and_tool_history",
}
AGENT_BUDGET_SHA256=stable_json_sha256(AGENT_BUDGET)
MEASUREMENT_CONTRACT_SHA256=stable_json_sha256(MEASUREMENT_CONTRACT)

ENGINE_RUNTIME_CONTRACT={
    "model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256,
    "runtime_stack_sha256":RUNTIME_STACK_SHA256,
    "engine_config_sha256":ENGINE_CONFIG_SHA256,
    "chat_template_sha256":template_sha,
    "tool_schema_sha256":TOOL_SCHEMA_SHA256,
}

THINK_CLOSE_IDS=tokenizer.encode("</think>",add_special_tokens=False)
EARLY_STOPPING_IDS=tokenizer.encode(QWEN_EARLY_STOPPING_TEXT,add_special_tokens=False)
if not THINK_CLOSE_IDS:
    raise RuntimeError("Tokenizer produced no token IDs for </think>")
if not EARLY_STOPPING_IDS:
    raise RuntimeError("Tokenizer produced no early-stopping token IDs")

def arm_root(arm):
    p=WORK_ROOT/arm.lower()
    p.mkdir(parents=True,exist_ok=True)
    (p/"traces").mkdir(exist_ok=True)
    return p

def arm_paths(arm):
    root=arm_root(arm)
    return {
        "root":root,
        "raw":root/"raw_generations.jsonl",
        "traces":root/"traces",
        "waves":root/"wave_metrics.jsonl",
        "runtime":root/"agent_loop_runtime.json",
        "checkpoint":root/"checkpoint_audit.json",
    }

def arm_sampling(arm):
    cfg=dict(BASE_SAMPLING)
    cfg["seed"]=int(ACT_BASE_SEED+int(ACT_ARMS[arm].get("seed_offset",0)))
    cfg["reasoning_effort"]=ACT_ARMS[arm]["reasoning_effort"]
    cfg["think_policy"]=ACT_ARMS[arm]["think_policy"]
    cfg["thinking_budget_tokens"]=ACT_ARMS[arm]["thinking_budget_tokens"]
    return cfg

def generation_contract_payload(arm,task_id):
    pr=PROMPT_RECORD_MAP_BY_ARM[arm][task_id]
    sampling=arm_sampling(arm)
    return {
        "generation_contract_version":GENERATION_CONTRACT_VERSION,
        "parser_contract_version":PARSER_CONTRACT_VERSION,
        "prompt_contract_version":PROMPT_CONTRACT_VERSION,
        "measurement_contract_sha256":MEASUREMENT_CONTRACT_SHA256,
        "experiment_id":EXPERIMENT_ID,"run_profile":RUN_PROFILE,"task_set":TASK_SET,
        "arm":arm,"arm_config":ACT_ARMS[arm],"task_id":task_id,
        "n_train":pr["n_train"],"n_test":pr["n_test"],
        "initial_prompt_sha256":pr["prompt_sha256"],"piv_source_commit":PIV_SOURCE_COMMIT,
        "sampling_config_sha256":stable_json_sha256(sampling),
        "agent_budget_sha256":AGENT_BUDGET_SHA256,
        **ENGINE_RUNTIME_CONTRACT,
    }

def current_contract_sha(arm,task_id):
    return stable_json_sha256(generation_contract_payload(arm,task_id))

def append_record(arm,rec):
    with open(arm_paths(arm)["raw"],"a",encoding="utf-8") as f:
        f.write(json.dumps(rec,ensure_ascii=False,default=str)+"\n")

def read_arm_records(arm):
    p=arm_paths(arm)["raw"]
    if not p.exists(): return []
    out=[]
    for i,line in enumerate(p.read_text(encoding="utf-8").splitlines()):
        if not line.strip(): continue
        try:
            r=json.loads(line); r["_checkpoint_order"]=i; out.append(r)
        except Exception: pass
    return out

def record_compatible(arm,r):
    tid=r.get("task_id")
    if r.get("arm")!=arm: return False,"arm_mismatch"
    if tid not in ACTIVE_TASK_SET: return False,"foreign_task"
    if r.get("generation_contract_sha256")!=current_contract_sha(arm,tid): return False,"contract_mismatch"
    return True,"ok"

def audit_arm_records(arm,records):
    compatible=[]; stale=[]; foreign=[]
    for r in records:
        ok,reason=record_compatible(arm,r)
        if ok: compatible.append(r)
        elif reason=="foreign_task": foreign.append({"task_id":r.get("task_id"),"record_id":r.get("record_id")})
        else: stale.append({"task_id":r.get("task_id"),"record_id":r.get("record_id"),"reason":reason})
    audit={"raw_rows":len(records),"compatible_rows":len(compatible),"stale_rows":len(stale),"foreign_rows":len(foreign),
           "stale_preview":stale[:20],"foreign_preview":foreign[:20]}
    return compatible,audit

def terminal_completed_task_ids(arm,records):
    out=set()
    for r in records:
        ok,_=record_compatible(arm,r)
        if ok and r.get("status")=="complete": out.add(r["task_id"])
    return out

def save_event(arm,task_id,event):
    p=arm_paths(arm)["traces"]/f"{task_id}.jsonl"
    with open(p,"a",encoding="utf-8") as f:
        f.write(json.dumps(event,ensure_ascii=False,default=str)+"\n")

def append_wave_metric(arm,row):
    with open(arm_paths(arm)["waves"],"a",encoding="utf-8") as f:
        f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n")

def assistant_tool_message(parsed):
    calls=[]
    for tc in parsed.get("tool_calls",[]):
        calls.append({"type":"function","function":{"name":tc["name"],"arguments":tc["arguments"]}})
    # parse_agent_turn already removes the literal closing tag from reasoning_content.
    reasoning=parsed.get("reasoning_content","") or ""
    if "</think>" in reasoning:
        raise RuntimeError("History contract violation: literal </think> leaked into reasoning_content")
    return {"role":"assistant","content":parsed.get("content","") or "",
            "reasoning_content":reasoning,"tool_calls":calls}

def _find_subseq(seq,sub):
    if not sub or len(sub)>len(seq): return -1
    first=sub[0]
    for i,x in enumerate(seq):
        if x==first and seq[i:i+len(sub)]==sub: return i
    return -1

def generated_think_post_counts(token_ids,text):
    ids=list(token_ids or [])
    idx=_find_subseq(ids,THINK_CLOSE_IDS)
    if idx>=0:
        close_end=idx+len(THINK_CLOSE_IDS)
        return close_end,max(0,len(ids)-close_end),True
    # Fallback only if tokenizer-specific segmentation hides the exact sequence.
    if "</think>" in (text or ""):
        prefix=(text or "").split("</think>",1)[0]+"</think>"
        n=min(len(ids),len(tokenizer.encode(prefix,add_special_tokens=False)))
        return n,max(0,len(ids)-n),True
    return len(ids),0,False

def code_state_effects(state,code):
    """Deterministic proxy for whether later Python calls reuse symbols created in earlier calls."""
    try: tree=py_ast.parse(code)
    except Exception: return {"reuse":[],"defined":[]}
    loaded=set(); defined=set()
    for node in py_ast.walk(tree):
        if isinstance(node,py_ast.Name):
            if isinstance(node.ctx,py_ast.Load): loaded.add(node.id)
            elif isinstance(node.ctx,(py_ast.Store,py_ast.Param)): defined.add(node.id)
        elif isinstance(node,(py_ast.FunctionDef,py_ast.AsyncFunctionDef,py_ast.ClassDef)):
            defined.add(node.name)
        elif isinstance(node,py_ast.Import):
            for a in node.names: defined.add(a.asname or a.name.split(".")[0])
        elif isinstance(node,py_ast.ImportFrom):
            for a in node.names: defined.add(a.asname or a.name)
    prior=set(state.get("defined_symbols",[]))
    reuse=sorted(loaded & prior)
    state["defined_symbols"]=sorted(prior|defined)
    if reuse:
        state["stateful_reuse_events"]+=1
        state["stateful_reused_symbols"]=sorted(set(state["stateful_reused_symbols"])|set(reuse))
    return {"reuse":reuse,"defined":sorted(defined)}

def complete_record(arm,state,task,verifier,candidate_status,stop_reason,error=""):
    test_outputs=verifier.get("test_outputs",[]) if verifier else []
    code=state.get("final_code")
    sampling=arm_sampling(arm)
    tool_hashes=state.get("tool_code_hashes",[])
    rec={
        "run_id":RUN_ID,"record_id":str(uuid.uuid4()),"arm":arm,
        "reasoning_effort":ACT_ARMS[arm]["reasoning_effort"],
        "think_policy":ACT_ARMS[arm]["think_policy"],
        "thinking_budget_tokens":ACT_ARMS[arm]["thinking_budget_tokens"],
        "task_id":state["task_id"],"branch":"piv_act_v18","status":"complete",
        "candidate_status":candidate_status,"stop_reason":stop_reason,"error":error,
        "n_train":len(task["train"]),"n_test":len(task["test"]),"seed":sampling["seed"],
        "sampling_seeds":state.get("sampling_seeds",[]),
        "source_retries":state.get("source_retries",0),"step_retries_total":state.get("step_retries_total",0),
        "retry_reasons":state.get("retry_reasons",[]),"budget_denials":state.get("budget_denials",0),
        "guard_rejections":state.get("guard_rejections",0),"salvaged_final_on_length":state.get("salvaged_final_on_length",0),
        "model_turns":state["model_turns"],"tool_calls":state["tool_calls"],
        "tool_call_emitted_turns":state.get("tool_call_emitted_turns",0),
        "tool_successes":state.get("tool_successes",0),"tool_errors":state["tool_errors"],
        "tool_timeouts":state["tool_timeouts"],"tool_empty_output_successes":state.get("tool_empty_output_successes",0),
        "worker_restarts":state["worker_restarts"],"repeated_tool_calls":state.get("repeated_tool_calls",0),
        "unique_tool_code_count":len(set(tool_hashes)),
        "stateful_reuse_events":state.get("stateful_reuse_events",0),
        "stateful_reused_symbols":state.get("stateful_reused_symbols",[]),
        "generated_tokens":state["generated_tokens"],
        "think_generated_tokens":state.get("think_generated_tokens",0),
        "post_close_generated_tokens":state.get("post_close_generated_tokens",0),
        "injected_stop_tokens":state.get("injected_stop_tokens",0),
        "natural_think_close_count":state.get("natural_think_close_count",0),
        "forced_think_close_count":state.get("forced_think_close_count",0),
        "first_tool_turn":state.get("first_tool_turn"),
        "tokens_to_first_tool":state.get("tokens_to_first_tool"),
        "max_rendered_prompt_tokens":state["max_prompt_tokens"],"finish_reasons":state["finish_reasons"],
        "final_code":code,"final_code_length":len(code or ""),
        "final_code_sha256":hashlib.sha256((code or "").encode()).hexdigest() if code else None,
        "all_train_exact":bool(verifier and verifier.get("all_train_exact")),
        "test_execution_ok":bool(verifier and verifier.get("test_execution_ok")),
        "any_test_output_ok":bool(verifier and verifier.get("any_test_output_ok")),
        "test_errors":verifier.get("test_errors",[]) if verifier else [],
        "verifier_status":verifier.get("verifier_status") if verifier else None,
        "verifier_error":(verifier.get("train_error") or verifier.get("test_error") or verifier.get("error")) if verifier else None,
        "test_outputs":test_outputs,
        "initial_prompt_sha256":PROMPT_RECORD_MAP_BY_ARM[arm][state["task_id"]]["prompt_sha256"],
        "generation_contract_sha256":current_contract_sha(arm,state["task_id"]),
        "sampling_config_sha256":stable_json_sha256(sampling),
        "agent_budget_sha256":AGENT_BUDGET_SHA256,"measurement_contract_sha256":MEASUREMENT_CONTRACT_SHA256,
        "piv_source_commit":PIV_SOURCE_COMMIT,**ENGINE_RUNTIME_CONTRACT,
    }
    return rec

def step_retry_available(state):
    return (state.get("step_retry_count",0)<MAX_RETRIES
            and state.get("generated_tokens",0)<TOTAL_GENERATED_TOKEN_CAP
            and state.get("model_turns",0)<MAX_MODEL_TURNS)

def register_step_retry(state,reason):
    if not step_retry_available(state): return False
    state["step_retry_count"]=int(state.get("step_retry_count",0))+1
    state["step_retries_total"]=int(state.get("step_retries_total",0))+1
    state["sampling_retry_nonce"]=int(state.get("sampling_retry_nonce",0))+1
    state.setdefault("retry_reasons",[]).append(f"step:{reason}")
    return True

def _make_params(cap,seed):
    return SamplingParams(
        temperature=BASE_SAMPLING["temperature"],top_p=BASE_SAMPLING["top_p"],top_k=BASE_SAMPLING["top_k"],
        min_p=BASE_SAMPLING["min_p"],presence_penalty=BASE_SAMPLING["presence_penalty"],
        repetition_penalty=BASE_SAMPLING["repetition_penalty"],max_tokens=int(cap),ignore_eos=False,
        seed=int(seed),skip_special_tokens=False,spaces_between_special_tokens=False)

def _extract_req(req):
    if not getattr(req,"outputs",None):
        return {"ok":False,"text":"","token_ids":[],"finish_reason":"no_output"}
    o=req.outputs[0]
    return {"ok":True,"text":o.text or "","token_ids":list(getattr(o,"token_ids",[]) or []),
            "finish_reason":str(getattr(o,"finish_reason",None))}

def _projected_wave_sec(max_cap,runtime_ctx):
    samples=runtime_ctx["wave_spt_samples"]
    if samples:
        spt=max(float(DEFAULT_SEC_PER_OUTPUT_TOKEN),float(np.percentile(samples,90)))
    else:
        spt=float(DEFAULT_SEC_PER_OUTPUT_TOKEN)
    return float(max_cap)*spt*float(WAVE_TIME_SAFETY_FACTOR)+float(WAVE_OVERHEAD_SEC),spt

def _run_vllm_phase(arm,phase,prompts,caps,seeds,runtime_ctx,batch_no,task_ids):
    if not prompts: return [],0.0
    projected,spt=_projected_wave_sec(max(caps),runtime_ctx)
    if time.perf_counter()+projected>=deadline:
        return [{"deadline_projected":True,"projected_sec":projected,"spt":spt} for _ in prompts],0.0
    params=[_make_params(c,s) for c,s in zip(caps,seeds)]
    t0=time.perf_counter()
    outs=arc_llm.generate(prompts,params,use_tqdm=False)
    wave_sec=time.perf_counter()-t0
    extracted=[_extract_req(r) for r in outs]
    max_out=max([len(x["token_ids"]) for x in extracted] or [0])
    runtime_ctx["global_wave_no"]+=1
    append_wave_metric(arm,{
        "run_id":RUN_ID,"arm":arm,"phase":phase,"batch_no":batch_no,
        "global_arm_wave_no":runtime_ctx["global_wave_no"],"task_ids":list(task_ids),
        "active_count":len(task_ids),"caps":list(map(int,caps)),"request_seeds":list(map(int,seeds)),
        "output_tokens":[len(x["token_ids"]) for x in extracted],
        "finish_reasons":[x["finish_reason"] for x in extracted],
        "sum_output_tokens":int(sum(len(x["token_ids"]) for x in extracted)),
        "max_output_tokens":int(max_out),"wave_sec":float(wave_sec),
        "aggregate_output_tokens_per_sec":float(sum(len(x["token_ids"]) for x in extracted)/wave_sec) if wave_sec>0 else None,
        "sec_per_output_token_longest":float(wave_sec/max_out) if max_out>0 else None,
    })
    if max_out>0 and wave_sec>0:
        runtime_ctx["wave_spt_samples"].append(float(wave_sec/max_out))
    return extracted,wave_sec

def generate_logical_wave(arm,entries,runtime_ctx,batch_no):
    """One logical assistant turn per entry. BUDGET3072 may use two physical vLLM phases."""
    cfg=ACT_ARMS[arm]
    task_ids=[e["task_id"] for e in entries]
    prompts=[e["prompt"] for e in entries]
    caps=[e["cap"] for e in entries]
    seeds=[e["seed"] for e in entries]

    if cfg["think_policy"]!="qwen_two_stage_budget":
        outs,wsec=_run_vllm_phase(arm,"native",prompts,caps,seeds,runtime_ctx,batch_no,task_ids)
        results=[]
        for e,x in zip(entries,outs):
            if x.get("deadline_projected"):
                results.append(x); continue
            think_n,post_n,natural=generated_think_post_counts(x["token_ids"],x["text"])
            results.append({
                "ok":x["ok"],"text":x["text"],"token_ids":x["token_ids"],"finish_reason":x["finish_reason"],
                "generated_tokens":len(x["token_ids"]),"think_generated_tokens":think_n,
                "post_close_generated_tokens":post_n,"natural_think_close":natural,
                "forced_think_close":False,"injected_stop_tokens":0,
                "phase1_tokens":len(x["token_ids"]),"phase2_tokens":0,
                "phase1_finish_reason":x["finish_reason"],"phase2_finish_reason":None,
                "logical_wave_sec":wsec,
            })
        return results

    think_budget=int(cfg["thinking_budget_tokens"])
    p1_caps=[min(int(c),think_budget) for c in caps]
    p1,wsec1=_run_vllm_phase(arm,"think_phase",prompts,p1_caps,seeds,runtime_ctx,batch_no,task_ids)
    results=[None]*len(entries)
    p2_prompts=[]; p2_caps=[]; p2_seeds=[]; p2_task_ids=[]; p2_indices=[]; p2_meta=[]
    for i,(e,x) in enumerate(zip(entries,p1)):
        if x.get("deadline_projected"):
            results[i]=x; continue
        if not x["ok"]:
            results[i]={
                "ok":False,"text":"","token_ids":[],"finish_reason":"no_output","generated_tokens":0,
                "think_generated_tokens":0,"post_close_generated_tokens":0,"natural_think_close":False,
                "forced_think_close":False,"injected_stop_tokens":0,"phase1_tokens":0,"phase2_tokens":0,
                "phase1_finish_reason":"no_output","phase2_finish_reason":None,"logical_wave_sec":wsec1,
            }
            continue
        ids1=list(x["token_ids"]); text1=x["text"]; fr1=x["finish_reason"]
        think_n1,post_n1,natural=generated_think_post_counts(ids1,text1)
        # If phase 1 naturally terminated, it is already a complete logical turn.
        if str(fr1).lower()=="stop":
            results[i]={
                "ok":True,"text":text1,"token_ids":ids1,"finish_reason":fr1,
                "generated_tokens":len(ids1),"think_generated_tokens":think_n1,
                "post_close_generated_tokens":post_n1,"natural_think_close":natural,
                "forced_think_close":False,"injected_stop_tokens":0,
                "phase1_tokens":len(ids1),"phase2_tokens":0,
                "phase1_finish_reason":fr1,"phase2_finish_reason":None,"logical_wave_sec":wsec1,
            }
            continue

        remaining=int(e["cap"])-len(ids1)
        base_ids=tokenizer.encode(e["prompt"],add_special_tokens=False)
        forced=not natural
        inject_ids=list(EARLY_STOPPING_IDS) if forced else []
        continuation_ids=list(base_ids)+ids1+inject_ids
        safe_context=MAX_MODEL_LEN-len(continuation_ids)-CONTEXT_SAFETY_TOKENS
        p2cap=min(remaining,safe_context)
        if p2cap<1:
            composed=text1+(QWEN_EARLY_STOPPING_TEXT if forced else "")
            results[i]={
                "ok":True,"text":composed,"token_ids":ids1,"finish_reason":"length",
                "generated_tokens":len(ids1),"think_generated_tokens":think_n1,
                "post_close_generated_tokens":post_n1,"natural_think_close":natural,
                "forced_think_close":forced,"injected_stop_tokens":len(inject_ids),
                "phase1_tokens":len(ids1),"phase2_tokens":0,
                "phase1_finish_reason":fr1,"phase2_finish_reason":None,"logical_wave_sec":wsec1,
            }
            continue
        # Token-exact continuation follows Qwen's documented open-source thinking-budget technique.
        p2_prompts.append(TokensPrompt(prompt_token_ids=continuation_ids))
        # vLLM starts a fresh request for phase 2; use a deterministic phase-specific seed rather than
        # resetting the exact same RNG seed used by phase 1. This is logged as part of the BUDGET policy.
        p2_caps.append(int(p2cap)); p2_seeds.append(int(e["seed"]+1_000_000))
        p2_task_ids.append(e["task_id"]); p2_indices.append(i)
        p2_meta.append((ids1,text1,fr1,think_n1,post_n1,natural,forced,len(inject_ids)))
    if p2_prompts:
        p2,wsec2=_run_vllm_phase(arm,"post_close_phase",p2_prompts,p2_caps,p2_seeds,runtime_ctx,batch_no,p2_task_ids)
        for j,(idx,x) in enumerate(zip(p2_indices,p2)):
            ids1,text1,fr1,think_n1,post_n1,natural,forced,inject_n=p2_meta[j]
            if x.get("deadline_projected"):
                results[idx]=x; continue
            text2=x["text"]; ids2=list(x["token_ids"])
            composed=text1+(QWEN_EARLY_STOPPING_TEXT if forced else "")+text2
            # Only model-generated tokens count against the matched 12,288 budget.
            total_ids=ids1+ids2
            if forced:
                think_n=think_n1
                post_n=len(ids2)+post_n1
            else:
                think_n,_,_=generated_think_post_counts(ids1,text1)
                post_n=max(0,len(total_ids)-think_n)
            results[idx]={
                "ok":x["ok"],"text":composed,"token_ids":total_ids,"finish_reason":x["finish_reason"],
                "generated_tokens":len(total_ids),"think_generated_tokens":think_n,
                "post_close_generated_tokens":post_n,"natural_think_close":natural,
                "forced_think_close":forced,"injected_stop_tokens":inject_n,
                "phase1_tokens":len(ids1),"phase2_tokens":len(ids2),
                "phase1_finish_reason":fr1,"phase2_finish_reason":x["finish_reason"],
                "logical_wave_sec":wsec1+wsec2,
            }
    return results

def run_real_piv_arm(arm,fixed_setup_elapsed_sec):
    paths=arm_paths(arm)
    loop_t0=time.perf_counter()
    runtime_ctx={"global_wave_no":0,"wave_spt_samples":[]}
    existing=read_arm_records(arm); completed=terminal_completed_task_ids(arm,existing)
    pending=[t for t in ACTIVE_TASK_IDS if t not in completed]
    print(f"[{arm}] resume completed={len(completed)} pending={len(pending)}")

    for batch_no,start_idx in enumerate(range(0,len(pending),AGENT_BATCH_SIZE),1):
        tids=pending[start_idx:start_idx+AGENT_BATCH_SIZE]
        workers={}; states={}

        def start_fresh_attempt(tid,st,reason=None):
            if reason is not None:
                if st["source_retries"]>=MAX_RETRIES: return False
                if st["generated_tokens"]>=TOTAL_GENERATED_TOKEN_CAP or st["model_turns"]>=MAX_MODEL_TURNS: return False
                st["source_retries"]+=1; st["sampling_retry_nonce"]+=1
                st["retry_reasons"].append(str(reason))
            old=workers.get(tid)
            if old is not None:
                st["worker_restart_offset"]+=int(getattr(old,"restart_count",0))
                st["worker_restarts"]=st["worker_restart_offset"]
                try: old.close()
                except Exception: pass
            workers[tid]=PersistentPythonWorker(task_without_test_solutions(eval_challenges[tid]),TOOL_TIMEOUT_SEC)
            st["messages"]=copy.deepcopy(PROMPT_RECORD_MAP_BY_ARM[arm][tid]["initial_messages"])
            st["attempt_turns"]=0; st["step_retry_count"]=0; st["budget_nudged"]=False; st["final_code"]=None
            return True

        try:
            for tid in tids:
                states[tid]={
                    "task_id":tid,"messages":[],"model_turns":0,"attempt_turns":0,
                    "tool_calls":0,"tool_call_emitted_turns":0,"tool_successes":0,
                    "tool_errors":0,"tool_timeouts":0,"tool_empty_output_successes":0,
                    "worker_restarts":0,"worker_restart_offset":0,
                    "generated_tokens":0,"think_generated_tokens":0,"post_close_generated_tokens":0,
                    "injected_stop_tokens":0,"natural_think_close_count":0,"forced_think_close_count":0,
                    "first_tool_turn":None,"tokens_to_first_tool":None,
                    "max_prompt_tokens":0,"finish_reasons":[],"done":False,"final_code":None,
                    "source_retries":0,"step_retry_count":0,"step_retries_total":0,"sampling_retry_nonce":0,
                    "sampling_seeds":[],"retry_reasons":[],"budget_nudged":False,"budget_denials":0,
                    "guard_rejections":0,"salvaged_final_on_length":0,
                    "tool_code_hashes":[],"defined_symbols":[],"stateful_reuse_events":0,
                    "stateful_reused_symbols":[],
                }
                start_fresh_attempt(tid,states[tid])

            while True:
                active=[t for t in tids if not states[t]["done"]]
                if not active: break
                if time.perf_counter()>=deadline:
                    for tid in active:
                        st=states[tid]; st["done"]=True
                        rec=complete_record(arm,st,eval_challenges[tid],None,"failed","global_deadline","safety reserve reached")
                        rec["status"]="failed"; append_record(arm,rec)
                    break

                entries=[]
                for tid in active:
                    st=states[tid]
                    if st["model_turns"]>=MAX_MODEL_TURNS or st["generated_tokens"]>=TOTAL_GENERATED_TOKEN_CAP:
                        st["done"]=True
                        append_record(arm,complete_record(arm,st,eval_challenges[tid],None,"rejected","agent_budget_exhausted"))
                        continue
                    effort=ACT_ARMS[arm]["reasoning_effort"]
                    prompt,method=render_agent_chat(st["messages"],effort)
                    pt=len(tokenizer.encode(prompt,add_special_tokens=False))
                    st["max_prompt_tokens"]=max(st["max_prompt_tokens"],pt)
                    safe=MAX_MODEL_LEN-pt-CONTEXT_SAFETY_TOKENS
                    remaining=TOTAL_GENERATED_TOKEN_CAP-st["generated_tokens"]
                    cap=min(PER_TURN_MAX_TOKENS,remaining,safe)
                    if cap<32:
                        st["done"]=True
                        append_record(arm,complete_record(arm,st,eval_challenges[tid],None,"rejected","context_or_token_budget_exhausted"))
                        continue
                    seed=int(arm_sampling(arm)["seed"]+st["sampling_retry_nonce"])
                    st["sampling_seeds"].append(seed)
                    entries.append({"task_id":tid,"prompt":prompt,"cap":int(cap),"seed":seed})

                if not entries: continue
                logical=generate_logical_wave(arm,entries,runtime_ctx,batch_no)
                if any(x.get("deadline_projected") for x in logical):
                    for e,x in zip(entries,logical):
                        tid=e["task_id"]; st=states[tid]
                        if x.get("deadline_projected"):
                            st["done"]=True
                            rec=complete_record(
                                arm,st,eval_challenges[tid],None,"failed","global_deadline_projected",
                                f"projected physical wave {x.get('projected_sec',0):.0f}s would cross safety deadline"
                            )
                            rec["status"]="failed"; append_record(arm,rec)
                    # Do not start another wave after a projected global-deadline stop.
                    break

                for e,res in zip(entries,logical):
                    tid=e["task_id"]; st=states[tid]
                    st["model_turns"]+=1; st["attempt_turns"]+=1
                    nt=int(res.get("generated_tokens",0))
                    st["generated_tokens"]+=nt
                    st["think_generated_tokens"]+=int(res.get("think_generated_tokens",0))
                    st["post_close_generated_tokens"]+=int(res.get("post_close_generated_tokens",0))
                    st["injected_stop_tokens"]+=int(res.get("injected_stop_tokens",0))
                    st["natural_think_close_count"]+=int(bool(res.get("natural_think_close")))
                    st["forced_think_close_count"]+=int(bool(res.get("forced_think_close")))
                    fr=str(res.get("finish_reason","no_output")); st["finish_reasons"].append(fr)
                    text=res.get("text","") or ""
                    parsed=parse_agent_turn(text)
                    event={
                        "run_id":RUN_ID,"arm":arm,"task_id":tid,"turn":st["model_turns"],
                        "attempt_turn":st["attempt_turns"],"source_retry_index":st["source_retries"],
                        "prompt_tokens":st["max_prompt_tokens"],"generated_tokens":nt,"finish_reason":fr,
                        "raw_text":text,"parsed_kind":parsed["kind"],"request_seed":st["sampling_seeds"][-1],
                        "natural_think_close":bool(res.get("natural_think_close")),
                        "forced_think_close":bool(res.get("forced_think_close")),
                        "think_generated_tokens":int(res.get("think_generated_tokens",0)),
                        "post_close_generated_tokens":int(res.get("post_close_generated_tokens",0)),
                        "injected_stop_tokens":int(res.get("injected_stop_tokens",0)),
                        "phase1_tokens":int(res.get("phase1_tokens",0)),"phase2_tokens":int(res.get("phase2_tokens",0)),
                        "phase1_finish_reason":res.get("phase1_finish_reason"),
                        "phase2_finish_reason":res.get("phase2_finish_reason"),
                    }

                    if not res.get("ok",False):
                        if register_step_retry(st,"no_output"):
                            event["step_retry"]=True; save_event(arm,tid,event); continue
                        if start_fresh_attempt(tid,st,"no_output"):
                            event["source_retry"]=True; save_event(arm,tid,event); continue
                        st["done"]=True
                        rec=complete_record(arm,st,eval_challenges[tid],None,"failed","no_output","vLLM returned no output")
                        rec["status"]="failed"; append_record(arm,rec); save_event(arm,tid,event); continue

                    _salvaged=salvage_final_code_on_length(text,parsed) if fr.lower()=="length" else None
                    if _salvaged is not None:
                        st["salvaged_final_on_length"]+=1; event["salvaged_final_on_length"]=True
                    if not generation_finish_reason_ok(fr) and _salvaged is None:
                        if register_step_retry(st,f"invalid_finish_reason:{fr}"):
                            event["step_retry"]=True; save_event(arm,tid,event); continue
                        if start_fresh_attempt(tid,st,f"invalid_finish_reason:{fr}"):
                            event["source_retry"]=True; save_event(arm,tid,event); continue
                        st["done"]=True
                        append_record(arm,complete_record(
                            arm,st,eval_challenges[tid],None,"rejected","invalid_finish_reason",
                            f"finish_reason={fr}; retries exhausted or budget unavailable"
                        ))
                        save_event(arm,tid,event); continue

                    if parsed["kind"]=="malformed_tool":
                        if register_step_retry(st,"malformed_tool_xml"):
                            event["step_retry"]=True; save_event(arm,tid,event); continue
                        if start_fresh_attempt(tid,st,"malformed_tool_xml"):
                            event["source_retry"]=True; save_event(arm,tid,event); continue
                        st["tool_errors"]+=1; st["done"]=True
                        append_record(arm,complete_record(arm,st,eval_challenges[tid],None,"rejected","malformed_tool_xml"))
                        save_event(arm,tid,event); continue

                    if parsed["kind"]=="tool":
                        st["tool_call_emitted_turns"]+=1
                        calls=parsed.get("tool_calls",[])
                        malformed=any(tc.get("name")!="python" or "code" not in tc.get("arguments",{}) for tc in calls)
                        if malformed:
                            if register_step_retry(st,"malformed_or_unsupported_structured_tool"):
                                event["step_retry"]=True; save_event(arm,tid,event); continue
                            st["step_retry_count"]=0
                            st["messages"].append(assistant_tool_message(parsed))
                            for tc in calls:
                                st["messages"].append({"role":"tool","content":
                                    f"Tool call rejected: expected python(code=...). Got name={tc.get('name')!r}, keys={list(tc.get('arguments',{}))}."})
                            st["tool_errors"]+=1
                            event["tool_result"]="unsupported_or_invalid_structured_tool_call"
                            save_event(arm,tid,event); continue

                        st["step_retry_count"]=0
                        st["messages"].append(assistant_tool_message(parsed))
                        event_calls=[]
                        denial=("LOCAL P0 tool budget exhausted. No further Python execution is available. "
                                "Return the self-contained final `solution(grid)` Python code block now.")
                        for tc in calls:
                            if st["tool_calls"]>=MAX_TOOL_CALLS:
                                st["messages"].append({"role":"tool","content":denial})
                                st["budget_denials"]+=1
                                event_calls.append({"budget_denied":True,"tool_ok":False,
                                                    "tool_code":tc.get("arguments",{}).get("code"),
                                                    "tool_output_to_model":denial})
                                continue
                            code=tc["arguments"]["code"]
                            h=hashlib.sha256(code.strip().encode()).hexdigest()
                            if h in st["tool_code_hashes"]: st["repeated_tool_calls"]+=1
                            st["tool_code_hashes"].append(h)
                            reuse=code_state_effects(st,code)
                            rr=workers[tid].execute(code)
                            if rr.get("guard_rejected"):
                                st["guard_rejections"]+=1
                            else:
                                st["tool_calls"]+=1
                                if st["first_tool_turn"] is None:
                                    st["first_tool_turn"]=st["model_turns"]
                                    st["tokens_to_first_tool"]=st["generated_tokens"]
                            current_restarts=int(getattr(workers[tid],"restart_count",0))
                            st["worker_restarts"]=st["worker_restart_offset"]+current_restarts
                            if rr.get("ok") and not rr.get("guard_rejected"):
                                st["tool_successes"]+=1
                                if not str(rr.get("text","")).strip(): st["tool_empty_output_successes"]+=1
                            if not rr.get("ok"): st["tool_errors"]+=1
                            if rr.get("timeout"): st["tool_timeouts"]+=1
                            raw_tool=str(rr.get("text",""))[:MAX_TOOL_RAW_ARTIFACT_CHARS]
                            model_tool=sanitize_tool_output(compact_tool_output(raw_tool))
                            if st["tool_calls"]>=MAX_TOOL_CALLS:
                                model_tool+=("\n\n[LOCAL P0 BUDGET] This was the final permitted Python execution. "
                                             "Your next assistant response must be the final self-contained `solution(grid)` code block.")
                            st["messages"].append({"role":"tool","content":model_tool})
                            event_calls.append({"tool_code":code,"tool_code_sha256":h,"tool_ok":rr.get("ok"),
                                                "guard_rejected":rr.get("guard_rejected",False),
                                                "stateful_reuse":reuse["reuse"],
                                                "tool_raw_output":raw_tool,"tool_output_to_model":model_tool})
                        event["tool_calls_executed"]=event_calls
                        denied=sum(1 for x in event_calls if x.get("budget_denied"))
                        if denied:
                            event["tool_result"]="partial_tool_budget_denial_finalization_nudge"
                            if st["budget_nudged"] and st["tool_calls"]>=MAX_TOOL_CALLS:
                                st["done"]=True
                                append_record(arm,complete_record(
                                    arm,st,eval_challenges[tid],None,"rejected","tool_budget_exhausted_after_nudge"))
                            else:
                                st["budget_nudged"]=True
                        save_event(arm,tid,event); continue

                    if parsed["kind"]=="final" or _salvaged is not None:
                        st["step_retry_count"]=0
                        code=_salvaged if _salvaged is not None else parsed["final_code"]
                        st["final_code"]=code
                        verifier=fresh_verify_solution(code,eval_challenges[tid],VERIFIER_TIMEOUT_SEC)
                        if verifier.get("all_train_exact") and verifier.get("test_execution_ok"): cand="verified"
                        elif verifier.get("all_train_exact") and verifier.get("any_test_output_ok"): cand="verified_partial_test"
                        elif verifier.get("all_train_exact"): cand="test_execution_failed"
                        else: cand="train_rejected"
                        st["done"]=True; event["verifier_status"]=verifier.get("verifier_status")
                        append_record(arm,complete_record(arm,st,eval_challenges[tid],verifier,cand,"final_program"))
                        save_event(arm,tid,event); continue

                    if start_fresh_attempt(tid,st,"no_tool_call_or_final_solution"):
                        event["source_retry"]=True; save_event(arm,tid,event); continue
                    st["done"]=True
                    append_record(arm,complete_record(
                        arm,st,eval_challenges[tid],None,"rejected","no_tool_call_or_final_solution"))
                    save_event(arm,tid,event)

                print(f"[{arm}] batch {batch_no}: active={sum(not states[t]['done'] for t in tids)}")
        finally:
            for w in workers.values():
                try: w.close()
                except Exception: pass

    loop_sec=time.perf_counter()-loop_t0
    with open(paths["runtime"],"w") as f:
        json.dump({"run_id":RUN_ID,"arm":arm,"fixed_setup_elapsed_sec":float(fixed_setup_elapsed_sec),
                   "loop_elapsed_sec":float(loop_sec),"agent_batch_size":AGENT_BATCH_SIZE,
                   "active_tasks":len(ACTIVE_TASK_IDS),"physical_vllm_waves":runtime_ctx["global_wave_no"]},f,indent=2)
    records=read_arm_records(arm); compatible,audit=audit_arm_records(arm,records)
    with open(paths["checkpoint"],"w") as f: json.dump(audit,f,indent=2)
    print(f"[{arm}] loop complete {loop_sec/60:.1f} min | checkpoint:",audit)

# ------------------------------ PREFLIGHT TESTS ---------------------------------------
def run_activation_policy_preflight_tests():
    # Exact forced-close representation: parsed reasoning history must not retain a literal closing tag.
    tool='<tool_call><function=python><parameter=code>puzzle.keys()</parameter></function></tool_call>'
    composed="reasoning prefix"+QWEN_EARLY_STOPPING_TEXT+tool
    parsed=parse_agent_turn(composed)
    assert parsed["kind"]=="tool" and "puzzle.keys" in parsed["tool_args"]["code"]
    msg=assistant_tool_message(parsed)
    assert "</think>" not in msg["reasoning_content"]
    assert "Considering the limited time" in msg["reasoning_content"]

    # Token-continuation construction has exact prefix + model-generated phase1 + injected early-stop IDs.
    base=[11,12,13]; phase1=[21,22,23]
    cont=base+phase1+list(EARLY_STOPPING_IDS)
    assert cont[:len(base)]==base and cont[len(base):len(base)+len(phase1)]==phase1
    assert len(cont)==len(base)+len(phase1)+len(EARLY_STOPPING_IDS)

    # Stateful reuse metric is deterministic.
    st={"defined_symbols":[],"stateful_reuse_events":0,"stateful_reused_symbols":[]}
    a=code_state_effects(st,"x=3\ndef helper(a): return a+x")
    b=code_state_effects(st,"y=helper(x)")
    assert "x" in b["reuse"] and "helper" in b["reuse"] and st["stateful_reuse_events"]==1

    # Fresh exact verifier remains authoritative after a forced-close final artifact.
    fake={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]}]}
    vr=fresh_verify_solution("def solution(grid):\n    return [row[:] for row in grid]",fake,timeout_sec=5)
    assert vr["all_train_exact"] and vr["test_execution_ok"] and vr["test_outputs"]==[[[2]]]

    payload={
        "ok":True,"effort_contract":QWEN_REASONING_EFFORT_CONTRACT,
        "early_stopping_token_count":len(EARLY_STOPPING_IDS),
        "forced_close_history_no_duplicate_tag":True,
        "token_exact_continuation_contract":True,
        "stateful_reuse_metric":True,"fresh_verifier":vr["verifier_status"],
    }
    with open(WORK_ROOT/"activation_policy_selftest.json","w") as f: json.dump(payload,f,indent=2)
    print("ACT-v1.8 activation-policy PREFLIGHT tests PASS:",payload)

def write_mock_preflight_records():
    for arm in ACT_ARM_ORDER:
        paths=arm_paths(arm)
        for tid in ACTIVE_TASK_IDS:
            st={
                "task_id":tid,"model_turns":0,"tool_calls":0,"tool_call_emitted_turns":0,"tool_successes":0,
                "tool_errors":0,"tool_timeouts":0,"tool_empty_output_successes":0,"worker_restarts":0,
                "generated_tokens":0,"think_generated_tokens":0,"post_close_generated_tokens":0,
                "injected_stop_tokens":0,"natural_think_close_count":0,"forced_think_close_count":0,
                "first_tool_turn":None,"tokens_to_first_tool":None,"max_prompt_tokens":PROMPT_RECORD_MAP_BY_ARM[arm][tid]["prompt_tokens"],
                "finish_reasons":["mock_preflight"],"final_code":None,"sampling_seeds":[],
                "source_retries":0,"step_retries_total":0,"retry_reasons":[],"budget_denials":0,
                "guard_rejections":0,"salvaged_final_on_length":0,"tool_code_hashes":[],
                "defined_symbols":[],"stateful_reuse_events":0,"stateful_reused_symbols":[],
                "repeated_tool_calls":0,
            }
            append_record(arm,complete_record(arm,st,eval_challenges[tid],None,"mock_rejected","mock_preflight"))
        compatible,audit=audit_arm_records(arm,read_arm_records(arm))
        with open(paths["checkpoint"],"w") as f: json.dump(audit,f,indent=2)

run_activation_policy_preflight_tests()
if not USE_MODEL:
    write_mock_preflight_records()
    print("PREFLIGHT mock records written for all arms.")
else:
    _fixed_setup_elapsed=time.perf_counter()-notebook_start
    for _arm in ACT_ARM_ORDER:
        run_real_piv_arm(_arm,_fixed_setup_elapsed)

print("Cell 8 complete. Arms:",ACT_ARM_ORDER)


In [ ]:
# =====================================================================================
# CELL 9 — THREE-ARM COVERAGE, SCORING, VERIFIED CANDIDATE REGISTRY
# =====================================================================================
latest_by_arm={}
coverage_by_arm={}
checkpoint_audit_by_arm={}
all_latest_rows=[]
all_score_rows=[]
registry=[]

weight_map=dict(zip(eval_weights["task_id"],eval_weights["weight"]))
quartile_map=dict(zip(eval_weights["task_id"],eval_weights["test_relevance_quartile"]))

for arm in ACT_ARM_ORDER:
    records=read_arm_records(arm)
    compatible,audit=audit_arm_records(arm,records)
    checkpoint_audit_by_arm[arm]=audit
    with open(arm_paths(arm)["checkpoint"],"w") as f: json.dump(audit,f,indent=2)

    latest_map={}
    for r in compatible:
        if r.get("status") in {"complete","failed"}:
            latest_map[r["task_id"]]=r
    latest_by_arm[arm]=latest_map
    complete_set={t for t,r in latest_map.items() if r.get("status")=="complete"}
    failed_set={t for t,r in latest_map.items() if r.get("status")=="failed"}
    missing=sorted(ACTIVE_TASK_SET-set(latest_map))
    coverage={
        "arm":arm,"complete_tasks":len(complete_set),"failed_tasks":len(failed_set),
        "expected_tasks":len(ACTIVE_TASK_IDS),"missing_tasks":missing,
        "terminal_coverage_complete":len(missing)==0 and len(latest_map)==len(ACTIVE_TASK_IDS),
        "run_complete":len(missing)==0 and len(failed_set)==0,
    }
    coverage_by_arm[arm]=coverage

    for tid in ACTIVE_TASK_IDS:
        r=latest_map.get(tid,{})
        if r: all_latest_rows.append(r)
        outs=r.get("test_outputs",[]) if r.get("all_train_exact") else []
        for qi,target in enumerate(eval_solutions[tid]):
            pred=outs[qi] if qi<len(outs) and is_rectangular_grid(outs[qi]) else None
            exact=int(pred is not None and grids_equal(pred,target))
            all_score_rows.append({
                "arm":arm,"task_id":tid,"test_index":qi,
                "piv_valid":int(pred is not None),"piv_exact":exact,
                "train_exact":int(bool(r.get("all_train_exact"))),
                "candidate_status":r.get("candidate_status"),"stop_reason":r.get("stop_reason"),
                "weight":weight_map[tid],"quartile":quartile_map[tid],
            })
            if pred is not None:
                grid_hash=hashlib.sha256(json.dumps(pred,separators=(",",":"),sort_keys=False).encode()).hexdigest()
                registry.append({
                    "arm":arm,"task_id":tid,"test_index":qi,"grid":pred,"canonical_sha256":grid_hash,
                    "source":"qwen3.8_27b_piv_act_v18","run_id":r.get("run_id"),"record_id":r.get("record_id"),
                    "seed":r.get("seed"),"program_sha256":r.get("final_code_sha256"),"train_exact":True,
                    "reasoning_effort":r.get("reasoning_effort"),"think_policy":r.get("think_policy"),
                    "piv_source_commit":PIV_SOURCE_COMMIT,
                    "generation_contract_sha256":r.get("generation_contract_sha256"),
                })

latest=pd.DataFrame(all_latest_rows)
scores=pd.DataFrame(all_score_rows)
registry_df=pd.DataFrame(registry)
scores.to_csv(WORK_ROOT/"task_output_scores_by_arm.csv",index=False)
registry_df.to_csv(WORK_ROOT/"candidate_registry.csv",index=False)
with open(WORK_ROOT/"coverage_by_arm.json","w") as f: json.dump(coverage_by_arm,f,indent=2)
with open(WORK_ROOT/"checkpoint_audit_by_arm.json","w") as f: json.dump(checkpoint_audit_by_arm,f,indent=2)

def weighted_mean(x,w):
    x=np.asarray(x,dtype=float); w=np.asarray(w,dtype=float)
    return float(np.sum(x*w)/np.sum(w)) if np.sum(w)>0 else float("nan")

mode_rows=[]
strict_rows=[]
for arm in ACT_ARM_ORDER:
    sub=scores[scores.arm==arm].copy()
    lm=latest_by_arm[arm]
    raw_acc=float(sub.piv_exact.mean()) if len(sub) else float("nan")
    weighted_acc=weighted_mean(sub.piv_exact,sub.weight) if len(sub) else float("nan")
    valid_rate=float(sub.piv_valid.mean()) if len(sub) else float("nan")
    train_rate=float(np.mean([bool(lm.get(tid,{}).get("all_train_exact")) for tid in ACTIVE_TASK_IDS]))
    task_tmp=sub.groupby("task_id")["piv_exact"].min()
    strict=float(task_tmp.mean()) if len(task_tmp) else float("nan")
    mode_rows.append({
        "arm":arm,"reasoning_effort":ACT_ARMS[arm]["reasoning_effort"],
        "think_policy":ACT_ARMS[arm]["think_policy"],
        "output_weighted_exact":raw_acc,"propensity_weighted_diagnostic":weighted_acc,
        "valid_grid_rate":valid_rate,"train_perfect_program_rate":train_rate,
        "strict_task_normalized_exact":strict,"tasks":len(ACTIVE_TASK_IDS),"test_outputs":len(sub),
    })
    for tid,val in task_tmp.items():
        strict_rows.append({"arm":arm,"task_id":tid,"piv_all_queries":int(val)})

mode_summary=pd.DataFrame(mode_rows)
strict_task_scores=pd.DataFrame(strict_rows)
mode_summary.to_csv(WORK_ROOT/"mode_summary.csv",index=False)
strict_task_scores.to_csv(WORK_ROOT/"strict_task_level_scores.csv",index=False)
print("Coverage:",json.dumps(coverage_by_arm,indent=2))
print(mode_summary.to_string(index=False))


In [ ]:
# =====================================================================================
# CELL 10 — EDA/SEGMENT EFFECTS BY ARM
# =====================================================================================
CATEGORY_MASKS={
    "ALL":lambda d:np.ones(len(d),dtype=bool),
    "tier1":lambda d:(d.proxy_tier==1).to_numpy(),
    "tier2":lambda d:(d.proxy_tier==2).to_numpy(),
    "tier3":lambda d:(d.proxy_tier==3).to_numpy(),
    "easy_static":lambda d:d.proxy_easy_static.astype(bool).to_numpy(),
    "rare":lambda d:d.proxy_rare.astype(bool).to_numpy(),
    "multi_query":lambda d:(d.proxy_n_queries>1).to_numpy(),
    "dynamic":lambda d:d.proxy_dynamic.astype(bool).to_numpy(),
    "high_ratio":lambda d:d.proxy_high_ratio.astype(bool).to_numpy(),
    "extreme_ratio":lambda d:d.proxy_extreme_ratio.astype(bool).to_numpy(),
}
cat_rows=[]
for arm in ACT_ARM_ORDER:
    sub_scores=scores[scores.arm==arm]
    score_task=sub_scores.groupby("task_id").agg(
        piv_exact=("piv_exact","mean"),valid=("piv_valid","mean"),train_exact=("train_exact","max")
    ).reset_index()
    diag_rows=[]
    for tid in ACTIVE_TASK_IDS:
        r=latest_by_arm[arm].get(tid,{})
        diag_rows.append({
            "task_id":tid,
            "tool_engaged":int(float(r.get("tool_calls",0) or 0)>0),
            "final_program":int(str(r.get("stop_reason",""))=="final_program"),
            "budget_death":int(str(r.get("stop_reason","")) in {
                "invalid_finish_reason","agent_budget_exhausted","context_or_token_budget_exhausted",
                "tool_budget_exhausted_after_nudge"
            }),
            "generated_tokens":float(r.get("generated_tokens",0) or 0),
        })
    diag=pd.DataFrame(diag_rows)
    cat=profile_task_table.merge(score_task,on="task_id",how="left").merge(diag,on="task_id",how="left")
    for name,fn in CATEGORY_MASKS.items():
        ss=cat.loc[fn(cat)]
        cat_rows.append({
            "arm":arm,"category":name,"tasks":len(ss),
            "mean_output_exact":float(ss.piv_exact.mean()) if len(ss) else np.nan,
            "mean_valid":float(ss.valid.mean()) if len(ss) else np.nan,
            "train_exact_task_rate":float(ss.train_exact.mean()) if len(ss) else np.nan,
            "tool_engagement_rate":float(ss.tool_engaged.mean()) if len(ss) else np.nan,
            "final_program_rate":float(ss.final_program.mean()) if len(ss) else np.nan,
            "budget_death_rate":float(ss.budget_death.mean()) if len(ss) else np.nan,
            "mean_generated_tokens":float(ss.generated_tokens.mean()) if len(ss) else np.nan,
        })
category_scores=pd.DataFrame(cat_rows)
category_scores.to_csv(WORK_ROOT/"category_scores_by_arm.csv",index=False)
display(category_scores)


In [ ]:
# =====================================================================================
# CELL 11 — PRE-REGISTERED EFFECT MATRIX + ARM GATES + FAILURE TAXONOMY
# =====================================================================================
BUDGET_DEATH_REASONS={
    "invalid_finish_reason","agent_budget_exhausted","context_or_token_budget_exhausted",
    "tool_budget_exhausted_after_nudge",
}
INFRA_FAILURE_REASONS={"no_output","global_deadline","global_deadline_projected"}

def _read_jsonl(path):
    out=[]
    if not path.exists(): return out
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        try: out.append(json.loads(line))
        except Exception: pass
    return out

def _distribution(vals):
    vals=[float(v) for v in vals if v is not None and not pd.isna(v)]
    if not vals: return {"count":0,"min":None,"median":None,"p90":None,"max":None,"mean":None,"values":[]}
    return {
        "count":len(vals),"min":float(np.min(vals)),"median":float(np.median(vals)),
        "p90":float(np.percentile(vals,90)),"max":float(np.max(vals)),"mean":float(np.mean(vals)),
        "values":[float(x) for x in vals],
    }

arm_effect_rows=[]
gate_by_arm={}
forced_close_rows=[]
transition_rows=[]
failure_rows=[]

for arm in ACT_ARM_ORDER:
    lm=latest_by_arm[arm]
    records=[lm.get(tid,{}) for tid in ACTIVE_TASK_IDS]
    stops=[str(r.get("stop_reason","")) for r in records]
    terminal_ids=[str(r.get("task_id")) for r in records if r.get("task_id") is not None]
    terminal_coverage=(len(records)==len(ACTIVE_TASK_IDS)
                       and len(set(terminal_ids))==len(ACTIVE_TASK_IDS)
                       and all(str(r.get("status","")) in {"complete","failed"} for r in records))
    budget_dead=[s in BUDGET_DEATH_REASONS for s in stops]
    infra_bad=[(s in INFRA_FAILURE_REASONS) or str(r.get("status",""))=="failed" for s,r in zip(stops,records)]
    tool_engaged=[float(r.get("tool_calls",0) or 0)>0 for r in records]
    tool_emitted=[float(r.get("tool_call_emitted_turns",0) or 0)>0 for r in records]
    final_program=[s=="final_program" for s in stops]
    train_exact=[bool(r.get("all_train_exact")) for r in records]
    test_ok=[bool(r.get("test_execution_ok")) for r in records]

    wave_rows=_read_jsonl(arm_paths(arm)["waves"])
    wave_secs=[float(w.get("wave_sec",0) or 0) for w in wave_rows]
    spts=[float(w["wave_sec"])/float(w["max_output_tokens"]) for w in wave_rows
          if w.get("max_output_tokens") and float(w.get("wave_sec",0) or 0)>0]
    rt={}
    if arm_paths(arm)["runtime"].exists():
        try: rt=json.loads(arm_paths(arm)["runtime"].read_text())
        except Exception: rt={}
    fixed=float(rt.get("fixed_setup_elapsed_sec",0) or 0)
    loop=float(rt.get("loop_elapsed_sec",0) or 0)
    scale=48/len(ACTIVE_TASK_IDS)
    projected=(fixed+loop*scale)/3600 if loop>0 else None
    projected20=(fixed+loop*scale*1.20)/3600 if loop>0 else None

    # Trace-derived forced-close outcomes and transition patterns.
    transition_counts={}
    fc_outcomes={"tool":0,"final":0,"text":0,"malformed_tool":0,"other":0}
    natural_outcomes={"tool":0,"final":0,"text":0,"malformed_tool":0,"other":0}
    for tid in ACTIVE_TASK_IDS:
        events=_read_jsonl(arm_paths(arm)["traces"]/f"{tid}.jsonl")
        seq=[]
        for ev in events:
            kind=str(ev.get("parsed_kind","other"))
            seq.append(kind)
            bucket=kind if kind in {"tool","final","text","malformed_tool"} else "other"
            if ev.get("forced_think_close"): fc_outcomes[bucket]+=1
            if ev.get("natural_think_close"): natural_outcomes[bucket]+=1
        for a,b in zip(seq,seq[1:]):
            k=f"{a}->{b}"; transition_counts[k]=transition_counts.get(k,0)+1
    forced_close_rows.append({"arm":arm,**{f"forced_to_{k}":v for k,v in fc_outcomes.items()},
                              **{f"natural_to_{k}":v for k,v in natural_outcomes.items()}})
    for k,v in sorted(transition_counts.items()):
        transition_rows.append({"arm":arm,"transition":k,"count":v})

    code_hashes=[r.get("final_code_sha256") for r in records if r.get("final_code_sha256")]
    unique_code=len(set(code_hashes))
    code_dup_ratio=(1-unique_code/len(code_hashes)) if code_hashes else np.nan
    arm_registry=registry_df[registry_df.arm==arm] if len(registry_df) else pd.DataFrame()
    grid_hashes=arm_registry["canonical_sha256"].tolist() if len(arm_registry) else []
    unique_grids=len(set(grid_hashes))
    grid_dup_ratio=(1-unique_grids/len(grid_hashes)) if grid_hashes else np.nan

    sub_mode=mode_summary[mode_summary.arm==arm].iloc[0]
    row={
        "arm":arm,"reasoning_effort":ACT_ARMS[arm]["reasoning_effort"],
        "think_policy":ACT_ARMS[arm]["think_policy"],
        "thinking_budget_tokens":ACT_ARMS[arm]["thinking_budget_tokens"],
        "terminal_coverage_complete":terminal_coverage,
        "infrastructure_failure_rate":float(np.mean(infra_bad)),
        "budget_death_rate":float(np.mean(budget_dead)),
        "tool_call_emitted_rate":float(np.mean(tool_emitted)),
        "tool_engagement_rate":float(np.mean(tool_engaged)),
        "final_program_rate":float(np.mean(final_program)),
        "all_train_exact_rate":float(np.mean(train_exact)),
        "test_execution_ok_rate":float(np.mean(test_ok)),
        "mean_generated_tokens":float(np.mean([float(r.get("generated_tokens",0) or 0) for r in records])),
        "mean_think_generated_tokens":float(np.mean([float(r.get("think_generated_tokens",0) or 0) for r in records])),
        "mean_post_close_generated_tokens":float(np.mean([float(r.get("post_close_generated_tokens",0) or 0) for r in records])),
        "mean_injected_stop_tokens":float(np.mean([float(r.get("injected_stop_tokens",0) or 0) for r in records])),
        "natural_think_close_rate":float(np.mean([float(r.get("natural_think_close_count",0) or 0)>0 for r in records])),
        "forced_think_close_rate":float(np.mean([float(r.get("forced_think_close_count",0) or 0)>0 for r in records])),
        "mean_tool_calls":float(np.mean([float(r.get("tool_calls",0) or 0) for r in records])),
        "mean_tool_successes":float(np.mean([float(r.get("tool_successes",0) or 0) for r in records])),
        "mean_tool_errors":float(np.mean([float(r.get("tool_errors",0) or 0) for r in records])),
        "mean_guard_rejections":float(np.mean([float(r.get("guard_rejections",0) or 0) for r in records])),
        "mean_repeated_tool_calls":float(np.mean([float(r.get("repeated_tool_calls",0) or 0) for r in records])),
        "stateful_reuse_task_rate":float(np.mean([float(r.get("stateful_reuse_events",0) or 0)>0 for r in records])),
        "tokens_to_first_tool_median":_distribution([r.get("tokens_to_first_tool") for r in records if r.get("tokens_to_first_tool") is not None])["median"],
        "source_retries_mean":float(np.mean([float(r.get("source_retries",0) or 0) for r in records])),
        "step_retries_mean":float(np.mean([float(r.get("step_retries_total",0) or 0) for r in records])),
        "salvaged_final_on_length_tasks":int(sum(float(r.get("salvaged_final_on_length",0) or 0)>0 for r in records)),
        "final_code_count":len(code_hashes),"unique_final_code_count":unique_code,
        "final_code_duplicate_ratio":code_dup_ratio,
        "candidate_grid_count":len(grid_hashes),"unique_candidate_grid_count":unique_grids,
        "candidate_grid_duplicate_ratio":grid_dup_ratio,
        "valid_grid_rate":float(sub_mode["valid_grid_rate"]),
        "output_weighted_exact":float(sub_mode["output_weighted_exact"]),
        "strict_task_normalized_exact":float(sub_mode["strict_task_normalized_exact"]),
        "physical_vllm_waves":len(wave_rows),"wave_sec_sum":float(sum(wave_secs)),
        "wave_sec_median":float(np.median(wave_secs)) if wave_secs else None,
        "sec_per_output_token_median":float(np.median(spts)) if spts else None,
        "projected_proxy48_total_hours":projected,
        "projected_proxy48_total_hours_20pct_margin":projected20,
    }
    arm_effect_rows.append(row)

    for tid,r in zip(ACTIVE_TASK_IDS,records):
        failure_rows.append({
            "arm":arm,"task_id":tid,"status":r.get("status"),"candidate_status":r.get("candidate_status"),
            "stop_reason":r.get("stop_reason"),"model_turns":r.get("model_turns"),"tool_calls":r.get("tool_calls"),
            "tool_call_emitted_turns":r.get("tool_call_emitted_turns"),"tool_successes":r.get("tool_successes"),
            "tool_errors":r.get("tool_errors"),"tool_timeouts":r.get("tool_timeouts"),
            "guard_rejections":r.get("guard_rejections"),"repeated_tool_calls":r.get("repeated_tool_calls"),
            "stateful_reuse_events":r.get("stateful_reuse_events"),"generated_tokens":r.get("generated_tokens"),
            "think_generated_tokens":r.get("think_generated_tokens"),"post_close_generated_tokens":r.get("post_close_generated_tokens"),
            "injected_stop_tokens":r.get("injected_stop_tokens"),"natural_think_close_count":r.get("natural_think_close_count"),
            "forced_think_close_count":r.get("forced_think_close_count"),"tokens_to_first_tool":r.get("tokens_to_first_tool"),
            "all_train_exact":r.get("all_train_exact"),"test_execution_ok":r.get("test_execution_ok"),
            "source_retries":r.get("source_retries"),"step_retries_total":r.get("step_retries_total"),
            "budget_denials":r.get("budget_denials"),"verifier_status":r.get("verifier_status"),
        })

    gate={
        "arm":arm,"terminal_coverage_complete":terminal_coverage,
        "infrastructure_failure_rate":float(np.mean(infra_bad)),
        "budget_death_rate":float(np.mean(budget_dead)),
        "tool_engagement_rate":float(np.mean(tool_engaged)),
        "final_program_rate":float(np.mean(final_program)),
        "all_train_exact_rate":float(np.mean(train_exact)),
        "projected_proxy48_total_hours":projected,
        "projected_proxy48_total_hours_20pct_margin":projected20,
    }
    gate["coverage_pass"]=bool(terminal_coverage)
    gate["infrastructure_pass"]=bool(gate["infrastructure_failure_rate"]==0.0)
    gate["budget_pass"]=bool(gate["budget_death_rate"]<=0.50)
    gate["activation_pass"]=bool(gate["tool_engagement_rate"]>=0.50)
    gate["productivity_pass"]=bool(gate["final_program_rate"]>=0.25)
    gate["wallclock_pass"]=bool(projected20 is not None and projected20<=10.0)
    gate["promotion_ready"]=bool(
        gate["coverage_pass"] and gate["infrastructure_pass"] and gate["budget_pass"]
        and gate["activation_pass"] and gate["productivity_pass"] and gate["wallclock_pass"]
    )
    gate["quality_strong_green"]=bool(gate["all_train_exact_rate"]>0)
    gate_by_arm[arm]=gate

arm_effects=pd.DataFrame(arm_effect_rows)
failure_df=pd.DataFrame(failure_rows)
forced_close_effects=pd.DataFrame(forced_close_rows)
transition_df=pd.DataFrame(transition_rows)

# Explicit deltas vs the frozen XHIGH activation-failure reference.
baseline=FROZEN_XHIGH_SMOKE_REFERENCE
for metric,base_key in [
    ("tool_engagement_rate","tool_engagement_rate"),
    ("final_program_rate","final_program_rate"),
    ("all_train_exact_rate","all_train_exact_rate"),
    ("budget_death_rate","budget_death_rate"),
    ("mean_generated_tokens","generated_tokens_per_task"),
]:
    arm_effects[f"delta_vs_xhigh_{metric}"]=arm_effects[metric]-float(baseline[base_key])

arm_effects.to_csv(WORK_ROOT/"arm_effect_matrix.csv",index=False)
failure_df.to_csv(WORK_ROOT/"agent_diagnostics_by_arm.csv",index=False)
forced_close_effects.to_csv(WORK_ROOT/"forced_close_effects.csv",index=False)
transition_df.to_csv(WORK_ROOT/"turn_transitions.csv",index=False)
with open(WORK_ROOT/"arm_promotion_gates.json","w") as f: json.dump(gate_by_arm,f,indent=2)

print("="*116)
print("ACT-v1.8 EFFECT MATRIX")
print("="*116)
display(arm_effects)
print("Promotion gates:",json.dumps(gate_by_arm,indent=2))

# Never auto-promote from PREFLIGHT/mock data.
if USE_MODEL:
    winners=[a for a,g in gate_by_arm.items() if g["promotion_ready"]]
    if winners:
        print("ACTIVATION/FEASIBILITY GATE PASS:",winners,
              "| quality still requires Proxy-48 before any solver-family promotion.")
    else:
        print("NO ARM PASSED activation+productivity+budget+wall-clock gates. Do not run Proxy-48 PIV.")


In [ ]:
# =====================================================================================
# CELL 12 — SOURCE-STYLE REPLAY PAYLOADS BY ARM
# =====================================================================================
pred_by_arm={}
for arm in ACT_ARM_ORDER:
    pred={}
    lm=latest_by_arm[arm]
    for tid in ACTIVE_TASK_IDS:
        r=lm.get(tid,{})
        outs=r.get("test_outputs",[]) if r.get("all_train_exact") else []
        pred[tid]=[]
        for qi in range(len(eval_challenges[tid]["test"])):
            candidate=outs[qi] if qi<len(outs) and is_rectangular_grid(outs[qi]) else None
            pred[tid].append({"attempt_1":candidate,"attempt_2":None})
    pred_by_arm[arm]=pred
    with open(WORK_ROOT/f"predictions_{arm.lower()}_replay.json","w") as f:
        json.dump(pred,f)
print("Wrote per-arm source-style replay payloads.")


In [ ]:
# =====================================================================================
# CELL 13 — REPLAY SELF-CHECKS + ARM DENOMINATOR INVARIANTS
# =====================================================================================
for arm in ACT_ARM_ORDER:
    pass1=[]
    for tid in ACTIVE_TASK_IDS:
        for qi,target in enumerate(eval_solutions[tid]):
            a=pred_by_arm[arm][tid][qi]["attempt_1"]
            pass1.append(int(a is not None and grids_equal(a,target)))
    replay=float(np.mean(pass1))
    scored=float(mode_summary.loc[mode_summary.arm==arm,"output_weighted_exact"].iloc[0])
    assert abs(replay-scored)<1e-12,(arm,replay,scored)
    assert len(pass1)==len(scores[scores.arm==arm]),(arm,len(pass1),len(scores[scores.arm==arm]))
    print(f"{arm} replay self-check PASS:",replay)
print("All arm denominators and replay payloads agree.")


In [ ]:
# =====================================================================================
# CELL 14 — ACT-v1.8 METADATA + SOURCE/BEHAVIOR BOUNDARIES
# =====================================================================================
total_elapsed_sec=time.perf_counter()-notebook_start

HARNESS_PROVEN_BY_PREFLIGHT=[
    "Qwen effort values xhigh/medium/low render; high is rejected by the mounted template",
    "tool XML parser and history replay representation",
    "documented forced-close text composes into a parseable post-think tool action",
    "token-exact continuation construction contract",
    "persistent REPL worker and state-reuse metric",
    "fresh independent exact-train verifier",
    "checkpoint arm isolation and scoring denominators",
]
MODEL_BEHAVIOR_MEASURED_ONLY_ON_GPU=[
    "natural thinking length","forced-close response behavior","actual tool engagement",
    "final-program generation","train-perfect program rate","ARC exactness","real wave timing",
    "stateful tool reuse by Qwen","budget-death behavior",
]

source_verification={
    "source_commit":PIV_SOURCE_COMMIT,
    "external_sources":PIV_EXTERNAL_SOURCES,
    "verified_source_files":PIV_SOURCE_FILES,
    "qwen_reasoning_effort_contract":QWEN_REASONING_EFFORT_CONTRACT,
    "qwen_thinking_budget_reference":{
        "method":"two-stage generation: generate to thinking budget; append documented early-stop text if thinking is unfinished; continue generation",
        "thinking_budget_tokens":ACT_ARMS["BUDGET3072"]["thinking_budget_tokens"],
        "early_stopping_text_sha256":hashlib.sha256(QWEN_EARLY_STOPPING_TEXT.encode()).hexdigest(),
        "early_stopping_token_count":len(EARLY_STOPPING_IDS),
         "accounting":"injected early-stop tokens are context/prefill tokens, not model-generated budget tokens",
        "phase2_seed_policy":"phase2 uses phase1 request seed + 1,000,000 because vLLM is a fresh request and cannot preserve Transformers RNG state across calls",
    },
    "source_faithful_piv_core":{
        "persistent_repl":True,"puzzle_preloaded":True,"tool_errors_to_model":True,
        "self_contained_solution":True,"fresh_train_verifier":True,"all_train_exact_gate":True,
        "test_outputs_from_execution":True,
    },
    "intentional_activation_ablation":{
        "LOW":"Qwen native low effort",
        "MEDIUM":"Qwen native medium effort",
        "BUDGET3072":"Qwen native xhigh + documented local two-stage thinking-budget continuation",
        "frozen_xhigh_reference":FROZEN_XHIGH_SMOKE_REFERENCE,
    },
    "harness_proven_by_preflight":HARNESS_PROVEN_BY_PREFLIGHT,
    "model_behavior_measured_only_on_gpu":MODEL_BEHAVIOR_MEASURED_ONLY_ON_GPU,
    "not_in_experiment":["reviewer","second lineage","PTQ repair","NVARC hints","judge","evolution","prefix-caching change"],
}
with open(WORK_ROOT/"source_level_verification.json","w") as f:
    json.dump(source_verification,f,indent=2)

run_metadata={
    "experiment":EXPERIMENT_ID,"run_profile":RUN_PROFILE,"task_set":TASK_SET,
    "accuracy_is_real":ACCURACY_IS_REAL,"run_id":RUN_ID,
    "active_task_ids":ACTIVE_TASK_IDS,"proxy48_ordered_sha256":PROXY48_ORDERED_SHA256,
    "piv_source_commit":PIV_SOURCE_COMMIT,
    "generation_contract_version":GENERATION_CONTRACT_VERSION,
    "parser_contract_version":PARSER_CONTRACT_VERSION,
    "prompt_contract_version":PROMPT_CONTRACT_VERSION,
    "measurement_contract":MEASUREMENT_CONTRACT,
    "measurement_contract_sha256":MEASUREMENT_CONTRACT_SHA256,
    "arms":ACT_ARMS,"frozen_xhigh_reference":FROZEN_XHIGH_SMOKE_REFERENCE,
    "seed":SEED,"model_path":str(MODEL_PATH),"model_type":model_cfg.get("model_type"),
    "runtime_stack":RUNTIME_STACK,"model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256,
    "engine_config_sha256":ENGINE_CONFIG_SHA256,"chat_template_sha256":template_sha,
    "tool_schema_sha256":TOOL_SCHEMA_SHA256,"agent_budget":AGENT_BUDGET,
    "agent_budget_sha256":AGENT_BUDGET_SHA256,
    "model_load_sec":model_load_sec,"warmup_sec":warmup_sec,"total_elapsed_sec":total_elapsed_sec,
    "max_model_len":MAX_MODEL_LEN,"coverage_by_arm":coverage_by_arm,
    "checkpoint_audit_by_arm":checkpoint_audit_by_arm,
    "arm_promotion_gates":gate_by_arm,
    "source_verification":source_verification,
}
with open(WORK_ROOT/"run_metadata.json","w") as f:
    json.dump(run_metadata,f,indent=2,default=str)

report_lines=[
    f"# {EXPERIMENT_ID} — Report for ChatGPT",
    "",
    f"- Profile: **{RUN_PROFILE}**",
    f"- Real accuracy: **{ACCURACY_IS_REAL}**",
    f"- Active tasks: **{len(ACTIVE_TASK_IDS)}**",
    f"- Arms: **{', '.join(ACT_ARM_ORDER)}**",
    f"- Model load sec: **{model_load_sec:.2f}**",
    f"- Warmup sec: **{warmup_sec:.2f}**",
    "",
    "## Interpretation boundary",
    "PREFLIGHT proves harness semantics. It does not predict real model behavior.",
    "ACT_SMOKE_GPU measures reasoning-to-action behavior on the real Qwen model.",
    "",
    "## Frozen XHIGH reference",
    "```json",json.dumps(FROZEN_XHIGH_SMOKE_REFERENCE,indent=2),"```",
    "",
    "## Arm gates",
    "```json",json.dumps(gate_by_arm,indent=2),"```",
]
(WORK_ROOT/"REPORT_FOR_CHATGPT.md").write_text("\n".join(report_lines),encoding="utf-8")
print("Wrote run_metadata.json, source_level_verification.json, REPORT_FOR_CHATGPT.md")


In [ ]:
# =====================================================================================
# CELL 15 — GENERATION SUMMARIES + MANIFEST + BUNDLE
# =====================================================================================
def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""): h.update(b)
    return h.hexdigest()

compact_cols=[c for c in [
    "run_id","record_id","arm","reasoning_effort","think_policy","thinking_budget_tokens",
    "task_id","status","candidate_status","stop_reason","seed","n_train","n_test",
    "model_turns","tool_call_emitted_turns","tool_calls","tool_successes","tool_errors","tool_timeouts",
    "guard_rejections","repeated_tool_calls","unique_tool_code_count","stateful_reuse_events",
    "source_retries","step_retries_total","budget_denials","generated_tokens","think_generated_tokens",
    "post_close_generated_tokens","injected_stop_tokens","natural_think_close_count","forced_think_close_count",
    "first_tool_turn","tokens_to_first_tool","max_rendered_prompt_tokens","all_train_exact",
    "test_execution_ok","any_test_output_ok","verifier_status","final_code_length","final_code_sha256",
    "generation_contract_sha256","sampling_config_sha256","agent_budget_sha256","measurement_contract_sha256",
    "model_artifact_fingerprint_sha256","runtime_stack_sha256","engine_config_sha256","piv_source_commit"
] if c in latest.columns]
latest[compact_cols].to_csv(WORK_ROOT/"generation_summary_by_arm.csv",index=False)

manifest=[]
for p in sorted(WORK_ROOT.rglob("*")):
    if p.is_file() and p.name!="MANIFEST.csv":
        manifest.append({"file":str(p.relative_to(WORK_ROOT)),"bytes":p.stat().st_size,"sha256":sha256_file(p)})
pd.DataFrame(manifest).to_csv(WORK_ROOT/"MANIFEST.csv",index=False)

ZIP_PATH=Path("/kaggle/working")/BUNDLE_NAME
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(WORK_ROOT.rglob("*")):
        if p.is_file(): zf.write(p,arcname=str(p.relative_to(WORK_ROOT)))

print("="*116)
print(EXPERIMENT_ID,RUN_PROFILE,"PIPELINE COMPLETE")
print("Bundle:",ZIP_PATH)
print("ZIP MB:",round(ZIP_PATH.stat().st_size/1024**2,2))
print("UPLOAD bundle + executed notebook back to ChatGPT.")


## Safe run sequence — PIV-Q27-ACT-v1.8

This notebook is an **activation ablation**, not the final Proxy-48 solver run.

It defaults to:

```python
RUN_PROFILE = "PREFLIGHT"
```

### 1) PREFLIGHT

CPU-safe. It validates:
- the mounted Qwen reasoning-effort contract (`xhigh`, `medium`, `low`; `high` must be rejected);
- parser/tool-history behavior;
- forced-close composition without duplicate `</think>` in replay history;
- token-exact continuation construction;
- persistent REPL + state-reuse instrumentation;
- fresh all-train exact verifier;
- arm-isolated checkpoint/scoring/bundle contracts.

PREFLIGHT does **not** measure whether the real model will use the tools.

Expected bundle: `PIV_Q27_ACT_V18_PREFLIGHT_BUNDLE.zip`.

### 2) ACT_SMOKE_GPU

After PREFLIGHT passes, change only:

```python
RUN_PROFILE = "ACT_SMOKE_GPU"
```

Use **4×L4, Internet Off**.

One resident Qwen3.8-27B model is loaded once. The same frozen stratified SMOKE8 tasks run sequentially through:

1. `LOW` — `reasoning_effort="low"`
2. `MEDIUM` — `reasoning_effort="medium"`
3. `BUDGET3072` — `reasoning_effort="xhigh"` plus Qwen's documented two-stage thinking-budget technique

All arms keep the same:
- 8 tasks and order;
- persistent Python REPL;
- fresh exact train verifier;
- sampling temperature/top-p/top-k/min-p/penalties;
- same base seed;
- max 6 executed Python calls/task;
- max 12 logical assistant turns/task;
- **12,288 model-generated tokens/task total**;
- same grid/sandbox/checkpoint rules.

For `BUDGET3072`, the early-stopping string is injected context, not model output. Its token count is logged separately.

### 3) Promotion decision

No arm is promoted just because it calls Python.

The pre-registered engineering gate requires:
- terminal coverage = 100%;
- infrastructure failure rate = 0;
- budget-death rate ≤ 50%;
- actual Python engagement ≥ 50%;
- final-program rate ≥ 25%;
- projected Proxy-48 runtime with 20% margin ≤ 10 hours.

`all_train_exact_rate > 0` is a strong-green quality signal on SMOKE8, but not a mandatory engineering gate.

Compare in this order:

**activation → artifact productivity → executable quality → compute → stability → EDA slices**

If all three arms fail activation/productivity, stop PIV and proceed to PTQ-Q27-T0 rather than adding more PIV complexity.

Expected bundle: `PIV_Q27_ACT_V18_SMOKE_GPU_BUNDLE.zip`.

Do not turn on prefix caching, change batching, add reviewer/PTQ/NVARC hints, or alter the model checkpoint inside this experiment.
